In [ ]:
!pip install ultralytics==8.4.182 --quiet
!wget https://github.com/ultralytics/yolov5/releases/download/v1.0/DOTAv1.zip -P /tmp/data/
!unzip -q /tmp/data/DOTAv1.zip -d /tmp/data/

In [ ]:
from PIL import Image
from matplotlib.patches import FancyArrowPatch, Rectangle
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from torchvision import models, transforms
from ultralytics import SAM, YOLO
import cv2
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import time
import torch
import torch.nn as nn
import pandas as pd

In [ ]:
# >>> REPLACE these three paths with what Kaggle shows you in the input panel <<<
DIOR_WEIGHTS_PATH = "/kaggle/input/notebooks/sowshan1/amapp-dior/runs/dior_full_v2/weights/best.pt"
DOTA_WEIGHTS_PATH = "/kaggle/input/notebooks/sowshan1/amapp-dota/checkpoint_backups/dota_last_backup.pt"
EUROSAT_WEIGHTS_PATH = "/kaggle/input/notebooks/sowshan1/amapp-eurosat/eurosat_checkpoints/best.pt"

model_dior = YOLO(DIOR_WEIGHTS_PATH)
model_dota = YOLO(DOTA_WEIGHTS_PATH)

print("DIOR model classes:", model_dior.names)
print("DOTA model classes:", model_dota.names)

In [ ]:
DIOR_CLASSES = [
    "airplane", "airport", "baseballfield", "basketballcourt", "bridge",
    "chimney", "dam", "expresswayservicearea", "expresswaytollstation",
    "golffield", "groundtrackfield", "harbor", "overpass", "ship",
    "stadium", "storagetank", "tenniscourt", "trainstation",
    "vehicle", "windmill"
]

SENSITIVITY_TABLE = {
    "airport": 1.0, "dam": 0.9, "harbor": 0.8, "trainstation": 0.7,
    "expresswaytollstation": 0.6, "storage-tank": 0.6, "chimney": 0.5,
    "expresswayservicearea": 0.5, "overpass": 0.4, "bridge": 0.4,
    "stadium": 0.3, "groundtrackfield": 0.3, "baseballfield": 0.2,
    "basketballcourt": 0.2, "tenniscourt": 0.2, "golffield": 0.2,
    "windmill": 0.1, "ship": 0.3, "vehicle": 0.2, "airplane": 0.4,
    "plane": 0.4, "large-vehicle": 0.2, "small-vehicle": 0.2,
    "helicopter": 0.5, "roundabout": 0.2, "soccer-ball-field": 0.2,
    "swimming-pool": 0.1, "baseball-diamond": 0.2, "tennis-court": 0.2,
    "basketball-court": 0.2, "ground-track-field": 0.3,
}


In [ ]:
# >>> Once Cell 2 prints model_dota.names, build this list in that exact index order.
# model_dota.names is usually a dict like {0: 'plane', 1: 'ship', ...} -- convert it directly: <<<
DOTA_CLASSES = [model_dota.names[i] for i in range(len(model_dota.names))]
print(DOTA_CLASSES)

In [ ]:
SUBREGION_CLASSES = {
    "airport", "harbor", "dam", "trainstation",
    "storagetank", "expresswaytollstation",
}

SENSITIVITY_TABLE_PARTS = {
    "airport": {
        "runway": 1.0, "terminal": 0.8, "parking": 0.5,
        "vegetation": 0.1, "other": 0.3,
    },
    "harbor": {
        "docking_area": 0.8, "vessel_cluster": 0.6,
        "shoreline": 0.3, "water": 0.1, "other": 0.3,
    },
    "dam": {
        "spillway_structure": 0.9, "reservoir_edge": 0.4,
        "water": 0.1, "other": 0.3,
    },
    "trainstation": {
        "platform_structure": 0.7, "rail_yard": 0.6,
        "surrounding_lot": 0.3, "other": 0.3,
    },
    "storagetank": {
        "tank_structure": 0.6, "containment_area": 0.4, "other": 0.3,
    },
    "expresswaytollstation": {
        "toll_structure": 0.6, "roadway": 0.3, "other": 0.3,
    },
}


def obb_corners_to_aabb(corners):
    xs = [p[0] for p in corners]
    ys = [p[1] for p in corners]
    return [min(xs), min(ys), max(xs), max(ys)]


def compute_exposure(bbox, image_shape):
    x1, y1, x2, y2 = bbox
    img_h, img_w = image_shape[:2]
    box_area = max(0, x2 - x1) * max(0, y2 - y1)
    clipped_x1, clipped_y1 = max(0, x1), max(0, y1)
    clipped_x2, clipped_y2 = min(img_w, x2), min(img_h, y2)
    clipped_area = max(0, clipped_x2 - clipped_x1) * max(0, clipped_y2 - clipped_y1)
    if box_area == 0:
        return 0.0
    return clipped_area / box_area


def compute_uniqueness(cls_name, all_detections_in_image):
    class_counts = {}
    for d in all_detections_in_image:
        class_counts[d[0]] = class_counts.get(d[0], 0) + 1
    if not class_counts:
        return 0.5
    max_count = max(class_counts.values())
    this_count = class_counts.get(cls_name, 1)
    return 1 - (this_count / max_count) if max_count > 0 else 0.5


def compute_context(bbox, all_detections_in_image, radius=200):
    x1, y1, x2, y2 = bbox
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    nearby_risk = 0.0
    for cls_name, conf, other_bbox in all_detections_in_image:
        ox1, oy1, ox2, oy2 = other_bbox
        ocx, ocy = (ox1 + ox2) / 2, (oy1 + oy2) / 2
        dist = ((cx - ocx) ** 2 + (cy - ocy) ** 2) ** 0.5
        if 0 < dist <= radius:
            nearby_risk += SENSITIVITY_TABLE.get(cls_name, 0.3)
    return min(1.0, nearby_risk / 3.0)


def compute_mapr(cls_name, confidence, exposure, uniqueness, context,
                  weights=(0.3, 0.25, 0.2, 0.15, 0.1)):
    w1, w2, w3, w4, w5 = weights
    sensitivity = SENSITIVITY_TABLE.get(cls_name, 0.3)
    return (w1 * sensitivity + w2 * confidence + w3 * exposure
            + w4 * uniqueness + w5 * context)


def select_operator(mapr_score):
    if mapr_score < 0.25: return "none"
    elif mapr_score < 0.5: return "blur"
    elif mapr_score < 0.75: return "pixelate"
    else: return "mask"

In [ ]:
def apply_operator(image, bbox, operator):
    if operator in ("preserve", "none"):
        return image
    protected = image.copy()
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(image.shape[1], x2), min(image.shape[0], y2)
    if x2 <= x1 or y2 <= y1:
        return protected
    roi = protected[y1:y2, x1:x2]
    if operator == "blur":
        protected[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (31, 31), 0)
    elif operator == "pixelate":
        small = cv2.resize(roi, (max(roi.shape[1] // 15, 1), max(roi.shape[0] // 15, 1)),
                            interpolation=cv2.INTER_LINEAR)
        protected[y1:y2, x1:x2] = cv2.resize(small, (roi.shape[1], roi.shape[0]),
                                              interpolation=cv2.INTER_NEAREST)
    elif operator == "mask":
        protected[y1:y2, x1:x2] = 0
    else:
        raise ValueError(f"Unknown operator: {operator}")
    return protected


def baseline_fixed_mask(image, detections):
    protected = image.copy()
    for cls_name, conf, bbox in detections:
        protected = apply_operator(protected, bbox, "mask")
    return protected


def baseline_single_attribute(image, detections):
    protected = image.copy()
    if not detections:
        return protected
    top = max(detections, key=lambda d: d[1])
    protected = apply_operator(protected, top[2], "mask")
    return protected

def apply_operator_obb(image, corners, operator):
    if operator in ("preserve", "none"):
        return image
    return apply_operator(image, obb_corners_to_aabb(corners), operator)

In [ ]:
def compute_residual_detectability(protected_image, model, class_list, original_count):
    tmp_path = "/tmp/_tmp_protected_dior.jpg"
    cv2.imwrite(tmp_path, protected_image)
    results = model.predict(source=tmp_path, save=False, conf=0.4, verbose=False)
    residual_count = 0
    for r in results:
        if r.boxes is not None:
            residual_count += len(r.boxes)
    return residual_count / original_count if original_count > 0 else 0.0

In [ ]:
# PART H — SUB-REGION SETUP (classes, per-part sensitivity)
# =============================================================================

SUBREGION_CLASSES = {
    "airport", "harbor", "dam", "trainstation",
    "storagetank", "expresswaytollstation",
}

SENSITIVITY_TABLE_PARTS = {
    "airport": {
        "runway": 1.0,
        "terminal": 0.8,
        "parking": 0.5,
        "vegetation": 0.1,
        "other": 0.3,
    },
    "harbor": {
        "docking_area": 0.8,
        "vessel_cluster": 0.6,
        "shoreline": 0.3,
        "water": 0.1,
        "other": 0.3,
    },
    "dam": {
        "spillway_structure": 0.9,
        "reservoir_edge": 0.4,
        "water": 0.1,
        "other": 0.3,
    },
    "trainstation": {
        "platform_structure": 0.7,
        "rail_yard": 0.6,
        "surrounding_lot": 0.3,
        "other": 0.3,
    },
    "storagetank": {
        "tank_structure": 0.6,
        "containment_area": 0.4,
        "other": 0.3,
    },
    "expresswaytollstation": {
        "toll_structure": 0.6,
        "roadway": 0.3,
        "other": 0.3,
    },
}


# =============================================================================
# PART I — SAM SEGMENTATION (box-prompted, multimask)
# =============================================================================

_sam_wrapper = None

_sam_predictor = None

def get_sam_model(checkpoint="sam_b.pt"):
    # bboxes=[...], multimask_output=True)`, matching how
    global _sam_wrapper, _sam_predictor
    if _sam_predictor is None:
        _sam_wrapper = SAM(checkpoint)
        dummy_image = np.zeros((64, 64, 3), dtype=np.uint8)
        _sam_wrapper.predict(dummy_image, bboxes=[[0, 0, 10, 10]], verbose=False)
        _sam_predictor = _sam_wrapper.predictor
    return _sam_predictor

def segment_detection_with_sam(image, bbox, sam_model=None):
    # internal parts. This function requests multimask_output=True via the
    # SyntaxError: 'multimask_output' is not a valid YOLO argument" -- the
    if sam_model is None:
        sam_model = get_sam_model()

    x1, y1, x2, y2 = [int(v) for v in bbox]
    results = sam_model(source=image, bboxes=[[x1, y1, x2, y2]], multimask_output=True)

    masks = []
    for r in results:
        if r.masks is None:
            continue
        for m in r.masks.data:
            mask_np = m.cpu().numpy().astype(bool)
            if mask_np.shape != image.shape[:2]:
                mask_np = cv2.resize(
                    mask_np.astype(np.uint8),
                    (image.shape[1], image.shape[0]),
                    interpolation=cv2.INTER_NEAREST,
                ).astype(bool)
            area = int(mask_np.sum())
            if area > 0:
                masks.append((mask_np, area))

    masks = _dedupe_masks(masks)
    masks.sort(key=lambda t: t[1], reverse=True)
    return masks

def _dedupe_masks(masks, iou_threshold=0.9):
    # multimask_output=True can return near-duplicate masks (e.g. two of the
    kept = []
    for mask, area in sorted(masks, key=lambda t: t[1], reverse=True):
        is_duplicate = False
        for kept_mask, _ in kept:
            intersection = np.logical_and(mask, kept_mask).sum()
            union = np.logical_or(mask, kept_mask).sum()
            iou = intersection / union if union > 0 else 0.0
            if iou > iou_threshold:
                is_duplicate = True
                break
        if not is_duplicate:
            kept.append((mask, area))
    return kept

def segment_detections_with_sam_batched(image, bboxes, sam_model=None):
    # Also applies Finding 3's multimask_output=True fix (see
    # concatenated in prompt order -- with multimask_output=True, that's
    if sam_model is None:
        sam_model = get_sam_model()
    if not bboxes:
        return []

    MASKS_PER_BOX = 3  # SAM's multimask_output=True candidate count

    int_boxes = [[int(v) for v in b] for b in bboxes]
    results = sam_model(source=image, bboxes=int_boxes, multimask_output=True)

    flat_masks = []
    for r in results:
        if r.masks is None:
            continue
        for m in r.masks.data:
            mask_np = m.cpu().numpy().astype(bool)
            if mask_np.shape != image.shape[:2]:
                mask_np = cv2.resize(
                    mask_np.astype(np.uint8),
                    (image.shape[1], image.shape[0]),
                    interpolation=cv2.INTER_NEAREST,
                ).astype(bool)
            flat_masks.append(mask_np)

    expected_total = len(int_boxes) * MASKS_PER_BOX
    if len(flat_masks) != expected_total:
        per_box_masks = []
        for bbox in bboxes:
            per_box_masks.append(segment_detection_with_sam(image, bbox, sam_model))
        return per_box_masks

    per_box_masks = []
    for i in range(len(int_boxes)):
        box_masks_raw = flat_masks[i * MASKS_PER_BOX:(i + 1) * MASKS_PER_BOX]
        box_masks = [(m, int(m.sum())) for m in box_masks_raw if m.sum() > 0]
        box_masks = _dedupe_masks(box_masks)
        box_masks.sort(key=lambda t: t[1], reverse=True)
        per_box_masks.append(box_masks)
    return per_box_masks


# =============================================================================
# PART J — PART LABELING: AIRPORT + GENERIC FALLBACK
# =============================================================================

def _mask_bbox_and_aspect(mask):
    # Return (width, height, aspect_ratio, fill_ratio) of a mask's
    # fill_ratio = mask area / bounding-box area (0-1) -- how much of its own
    # a runway) has HIGH fill_ratio (it densely fills a thin, long box). A
    # elongated airfield) has a low fill_ratio, since most of its own
    # FINDING 5 FIX: the original version of this function returned only
    # against real DIOR airport detections) to frequently mislabel the
    # the mask itself is actually a thin strip. Adding fill_ratio lets
    # fill_ratio) from "large blob that happens to sit in an elongated
    # outer box" (low fill_ratio) -- see label_parts_airport for how this is
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return 0, 0, 1.0, 0.0
    w = xs.max() - xs.min() + 1
    h = ys.max() - ys.min() + 1
    long_side, short_side = max(w, h), max(min(w, h), 1)
    aspect_ratio = long_side / short_side
    bbox_area = max(w * h, 1)
    fill_ratio = mask.sum() / bbox_area
    return w, h, aspect_ratio, fill_ratio

def label_parts_airport(masks, bbox):
    # mask (aspect ratio well above 1, AND high fill_ratio -- a thin strip
    # FINDING 5 FIX: previously used aspect_ratio alone, which was found (via
    # visual spot-check) to frequently mislabel the actual runway as
    # strip. Now requires BOTH high aspect ratio AND high fill_ratio for the
    labeled = []
    if not masks:
        return labeled

    total_box_area = max((bbox[2] - bbox[0]) * (bbox[3] - bbox[1]), 1)
    remaining = list(masks)

    # Runway: elongated (aspect >= 2.5) AND densely-filled (fill_ratio >=
    # box will have a much lower fill_ratio) and reasonably large (>3% of
    runway_candidates = [
        m for m in remaining
        if _mask_bbox_and_aspect(m[0])[2] >= 2.5
        and _mask_bbox_and_aspect(m[0])[3] >= 0.35
        and m[1] / total_box_area > 0.03
    ]
    if runway_candidates:
        # fill_ratio (most strip-like), not just the largest by area --
        # area alone was part of what caused Finding 5's mislabeling, since
        runway_mask = max(runway_candidates, key=lambda t: _mask_bbox_and_aspect(t[0])[3])
        labeled.append(("runway", runway_mask[0]))
        remaining = [m for m in remaining if m is not runway_mask]

    terminal_candidates = [
        m for m in remaining
        if not (_mask_bbox_and_aspect(m[0])[2] >= 2.5 and _mask_bbox_and_aspect(m[0])[3] >= 0.35)
        and m[1] / total_box_area > 0.03
    ]
    if terminal_candidates:
        terminal_mask = max(terminal_candidates, key=lambda t: t[1])
        labeled.append(("terminal", terminal_mask[0]))
        remaining = [m for m in remaining if m is not terminal_mask]

    parking_candidates = [m for m in remaining if m[1] / total_box_area > 0.01]
    if parking_candidates:
        parking_mask = max(parking_candidates, key=lambda t: t[1])
        labeled.append(("parking", parking_mask[0]))
        remaining = [m for m in remaining if m is not parking_mask]

    for mask, area in remaining:
        labeled.append(("vegetation", mask))

    return labeled

def label_parts_generic(masks, bbox, class_name):
    if not masks:
        return []
    part_names = list(SENSITIVITY_TABLE_PARTS.get(class_name, {}).keys())
    primary_name = part_names[0] if part_names else "other"

    labeled = [(primary_name, masks[0][0])]
    for mask, area in masks[1:]:
        labeled.append(("other", mask))
    return labeled

def label_parts(masks, bbox, class_name):
    dispatch = {
        "airport": label_parts_airport,
        "harbor": label_parts_harbor,
        "dam": label_parts_dam,
        "trainstation": label_parts_trainstation,
        "storagetank": label_parts_storagetank,
        "expresswaytollstation": label_parts_expresswaytollstation,
    }
    fn = dispatch.get(class_name)
    if fn is None:
        return label_parts_generic(masks, bbox, class_name)
    return fn(masks, bbox)


# =============================================================================
# PART K — SUB-REGION MAPR SCORING + MULTI-REGION PROTECTION
# =============================================================================

def get_part_sensitivity(class_name, part_name):
    class_parts = SENSITIVITY_TABLE_PARTS.get(class_name, {})
    if part_name in class_parts:
        return class_parts[part_name]
    return SENSITIVITY_TABLE.get(class_name, 0.3)

def compute_mapr_part_linear(part_sensitivity, confidence, exposure, uniqueness,
                              context, weights=(0.3, 0.25, 0.2, 0.15, 0.1)):
    w1, w2, w3, w4, w5 = weights
    return (w1 * part_sensitivity + w2 * confidence + w3 * exposure
            + w4 * uniqueness + w5 * context)

def score_all_parts(class_name, labeled_parts, confidence, exposure,
                     uniqueness, context):
    scored = []
    for part_name, mask in labeled_parts:
        part_sensitivity = get_part_sensitivity(class_name, part_name)
        mapr = compute_mapr_part_linear(part_sensitivity, confidence, exposure,
                                         uniqueness, context)
        operator = select_operator(mapr)
        scored.append((part_name, mask, mapr, operator))
    return scored

def apply_operator_to_mask(image, mask, operator):
    if operator in ("preserve", "none") or not mask.any():
        return image

    protected = image.copy()
    ys, xs = np.where(mask)
    y1, y2, x1, x2 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    roi = image[y1:y2, x1:x2]
    roi_mask = mask[y1:y2, x1:x2]

    if operator == "blur":
        transformed_roi = cv2.GaussianBlur(roi, (31, 31), 0)
    elif operator == "pixelate":
        small = cv2.resize(roi, (max(roi.shape[1] // 15, 1), max(roi.shape[0] // 15, 1)),
                            interpolation=cv2.INTER_LINEAR)
        transformed_roi = cv2.resize(small, (roi.shape[1], roi.shape[0]),
                                      interpolation=cv2.INTER_NEAREST)
    elif operator == "mask":
        transformed_roi = np.zeros_like(roi)
    else:
        raise ValueError(f"Unknown operator: {operator}")

    roi_out = roi.copy()
    roi_out[roi_mask] = transformed_roi[roi_mask]
    protected[y1:y2, x1:x2] = roi_out
    return protected

def apply_operator_multiregion(image, scored_parts):
    protected = image.copy()
    for part_name, mask, mapr, operator in sorted(scored_parts, key=lambda p: p[2], reverse=True):
        protected = apply_operator_to_mask(protected, mask, operator)
    return protected


# =============================================================================
# PART L — SUB-REGION PIPELINE (DIOR)
# =============================================================================

def run_amapp_subregion_linear(image, detections, img_shape, sam_model=None):
    if sam_model is None:
        sam_model = get_sam_model()

    protected = image.copy()
    per_detection_log = []

    subregion_indices = [
        i for i, (cls_name, _, _) in enumerate(detections)
        if cls_name in SUBREGION_CLASSES
    ]
    subregion_bboxes = [detections[i][2] for i in subregion_indices]
    batched_masks = segment_detections_with_sam_batched(protected, subregion_bboxes, sam_model)
    masks_by_index = dict(zip(subregion_indices, batched_masks))

    for det_idx, (cls_name, confidence, bbox) in enumerate(detections):
        exposure = compute_exposure(bbox, img_shape)
        uniqueness = compute_uniqueness(cls_name, detections)
        context = compute_context(bbox, detections)

        if cls_name in SUBREGION_CLASSES:
            raw_masks = masks_by_index.get(det_idx, [])
            labeled_parts = label_parts(raw_masks, bbox, cls_name)

            if not labeled_parts:
                mapr = compute_mapr(cls_name, confidence, exposure, uniqueness, context)
                operator = select_operator(mapr)
                protected = apply_operator(protected, bbox, operator)
                per_detection_log.append({
                    "class": cls_name, "path": "whole_object_fallback_no_masks",
                    "mapr": mapr, "operator": operator,
                })
                continue

            scored_parts = score_all_parts(cls_name, labeled_parts, confidence,
                                            exposure, uniqueness, context)
            protected = apply_operator_multiregion(protected, scored_parts)
            per_detection_log.append({
                "class": cls_name, "path": "subregion",
                "parts": [(p[0], p[2], p[3]) for p in scored_parts],
            })
        else:
            mapr = compute_mapr(cls_name, confidence, exposure, uniqueness, context)
            operator = select_operator(mapr)
            protected = apply_operator(protected, bbox, operator)
            per_detection_log.append({
                "class": cls_name, "path": "whole_object",
                "mapr": mapr, "operator": operator,
            })

    return protected, per_detection_log


# =============================================================================
# PART M — PART LABELING: HARBOR/DAM/TRAINSTATION/STORAGETANK/TOLLSTATION
# =============================================================================

def label_parts_harbor(masks, bbox):
    labeled = []
    if not masks:
        return labeled

    total_box_area = max((bbox[2] - bbox[0]) * (bbox[3] - bbox[1]), 1)
    remaining = list(masks)

    docking_candidates = [
        m for m in remaining
        if 1.8 <= _mask_bbox_and_aspect(m[0])[2] <= 4.0 and m[1] / total_box_area > 0.04
    ]
    if docking_candidates:
        docking_mask = max(docking_candidates, key=lambda t: t[1])
        labeled.append(("docking_area", docking_mask[0]))
        remaining = [m for m in remaining if m is not docking_mask]

    vessel_candidates = [
        m for m in remaining
        if _mask_bbox_and_aspect(m[0])[2] < 2.5 and 0.005 < m[1] / total_box_area < 0.05
    ]
    if vessel_candidates:
        combined_mask = np.zeros_like(vessel_candidates[0][0])
        for m, _ in vessel_candidates:
            combined_mask |= m
        labeled.append(("vessel_cluster", combined_mask))
        vessel_candidate_ids = {id(m) for m, _ in vessel_candidates}
        remaining = [m for m in remaining if id(m[0]) not in vessel_candidate_ids]

    if remaining:
        water_mask = max(remaining, key=lambda t: t[1])
        labeled.append(("water", water_mask[0]))
        remaining = [m for m in remaining if m is not water_mask]

    for mask, area in remaining:
        labeled.append(("shoreline", mask))

    return labeled

def label_parts_dam(masks, bbox):
    labeled = []
    if not masks:
        return labeled

    total_box_area = max((bbox[2] - bbox[0]) * (bbox[3] - bbox[1]), 1)
    remaining = list(masks)

    structure_candidates = [
        m for m in remaining
        if _mask_bbox_and_aspect(m[0])[2] < 2.0 and m[1] / total_box_area > 0.10
    ]
    if structure_candidates:
        structure_mask = max(structure_candidates, key=lambda t: t[1])
        labeled.append(("spillway_structure", structure_mask[0]))
        remaining = [m for m in remaining if m is not structure_mask]

    edge_candidates = [m for m in remaining if _mask_bbox_and_aspect(m[0])[2] >= 2.0]
    if edge_candidates:
        edge_mask = max(edge_candidates, key=lambda t: t[1])
        labeled.append(("reservoir_edge", edge_mask[0]))
        remaining = [m for m in remaining if m is not edge_mask]

    if remaining:
        water_mask = max(remaining, key=lambda t: t[1])
        labeled.append(("water", water_mask[0]))
        remaining = [m for m in remaining if m is not water_mask]

    for mask, area in remaining:
        labeled.append(("other", mask))

    return labeled

def label_parts_trainstation(masks, bbox):
    labeled = []
    if not masks:
        return labeled

    total_box_area = max((bbox[2] - bbox[0]) * (bbox[3] - bbox[1]), 1)
    remaining = list(masks)

    platform_candidates = [
        m for m in remaining
        if _mask_bbox_and_aspect(m[0])[2] < 2.2 and m[1] / total_box_area > 0.05
    ]
    if platform_candidates:
        platform_mask = max(platform_candidates, key=lambda t: t[1])
        labeled.append(("platform_structure", platform_mask[0]))
        remaining = [m for m in remaining if m is not platform_mask]

    rail_candidates = [
        m for m in remaining
        if _mask_bbox_and_aspect(m[0])[2] >= 2.2 and m[1] / total_box_area > 0.03
    ]
    if rail_candidates:
        rail_mask = max(rail_candidates, key=lambda t: t[1])
        labeled.append(("rail_yard", rail_mask[0]))
        remaining = [m for m in remaining if m is not rail_mask]

    lot_candidates = [m for m in remaining if m[1] / total_box_area > 0.01]
    if lot_candidates:
        lot_mask = max(lot_candidates, key=lambda t: t[1])
        labeled.append(("surrounding_lot", lot_mask[0]))
        remaining = [m for m in remaining if m is not lot_mask]

    for mask, area in remaining:
        labeled.append(("other", mask))

    return labeled

def label_parts_storagetank(masks, bbox):
    labeled = []
    if not masks:
        return labeled

    total_box_area = max((bbox[2] - bbox[0]) * (bbox[3] - bbox[1]), 1)
    remaining = list(masks)

    tank_candidates = [
        m for m in remaining
        if _mask_bbox_and_aspect(m[0])[2] < 1.3 and m[1] / total_box_area > 0.02
    ]
    if tank_candidates:
        combined_mask = np.zeros_like(tank_candidates[0][0])
        for m, _ in tank_candidates:
            combined_mask |= m
        labeled.append(("tank_structure", combined_mask))
        tank_candidate_ids = {id(m) for m, _ in tank_candidates}
        remaining = [m for m in remaining if id(m[0]) not in tank_candidate_ids]

    if remaining:
        containment_mask = max(remaining, key=lambda t: t[1])
        labeled.append(("containment_area", containment_mask[0]))
        remaining = [m for m in remaining if m is not containment_mask]

    for mask, area in remaining:
        labeled.append(("other", mask))

    return labeled

def label_parts_expresswaytollstation(masks, bbox):
    labeled = []
    if not masks:
        return labeled

    total_box_area = max((bbox[2] - bbox[0]) * (bbox[3] - bbox[1]), 1)
    remaining = list(masks)

    structure_candidates = [
        m for m in remaining
        if _mask_bbox_and_aspect(m[0])[2] < 2.0 and m[1] / total_box_area > 0.03
    ]
    if structure_candidates:
        structure_mask = max(structure_candidates, key=lambda t: t[1])
        labeled.append(("toll_structure", structure_mask[0]))
        remaining = [m for m in remaining if m is not structure_mask]

    roadway_candidates = [m for m in remaining if _mask_bbox_and_aspect(m[0])[2] >= 2.0]
    if roadway_candidates:
        roadway_mask = max(roadway_candidates, key=lambda t: t[1])
        labeled.append(("roadway", roadway_mask[0]))
        remaining = [m for m in remaining if m is not roadway_mask]

    for mask, area in remaining:
        labeled.append(("other", mask))

    return labeled


# =============================================================================
# PART M2 — LABEL DISPATCHER (overrides Part J's dispatcher for all 6 classes)
# =============================================================================

def label_parts(masks, bbox, class_name):
    dispatch = {
        "airport": label_parts_airport,
        "harbor": label_parts_harbor,
        "dam": label_parts_dam,
        "trainstation": label_parts_trainstation,
        "storagetank": label_parts_storagetank,
        "expresswaytollstation": label_parts_expresswaytollstation,
    }
    fn = dispatch.get(class_name)
    if fn is None:
        return label_parts_generic(masks, bbox, class_name)
    return fn(masks, bbox)


# =============================================================================
# PART N — SUB-REGION OBB SUPPORT (DOTA)
# =============================================================================

def polygon_to_mask(corners, img_shape):
    h, w = img_shape[:2]
    pts = np.array(corners, dtype=np.int32).reshape(-1, 2)
    poly_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(poly_mask, [pts], 1)
    return poly_mask.astype(bool)

def run_amapp_subregion_linear_obb(image, detections, img_shape, sam_model=None):
    if sam_model is None:
        sam_model = get_sam_model()

    protected = image.copy()
    per_detection_log = []

    subregion_indices = [
        i for i, (cls_name, _, _, _) in enumerate(detections)
        if cls_name in SUBREGION_CLASSES
    ]
    subregion_aabbs = [detections[i][2] for i in subregion_indices]
    batched_masks = segment_detections_with_sam_batched(protected, subregion_aabbs, sam_model)
    masks_by_index = dict(zip(subregion_indices, batched_masks))

    for det_idx, (cls_name, confidence, aabb, corners) in enumerate(detections):
        exposure = compute_exposure(aabb, img_shape)
        detections_view = [(d[0], d[1], d[2]) for d in detections]
        uniqueness = compute_uniqueness(cls_name, detections_view)
        context = compute_context(aabb, detections_view)

        if cls_name in SUBREGION_CLASSES:
            raw_masks = masks_by_index.get(det_idx, [])

            polygon_mask = polygon_to_mask(corners, img_shape)
            clipped_masks = []
            for mask, _ in raw_masks:
                clipped = mask & polygon_mask
                area = int(clipped.sum())
                if area > 0:
                    clipped_masks.append((clipped, area))
            clipped_masks.sort(key=lambda t: t[1], reverse=True)

            labeled_parts = label_parts(clipped_masks, aabb, cls_name)

            if not labeled_parts:
                mapr = compute_mapr(cls_name, confidence, exposure, uniqueness, context)
                operator = select_operator(mapr)
                protected = apply_operator_obb(protected, corners, operator)
                per_detection_log.append({
                    "class": cls_name, "path": "whole_object_fallback_no_masks",
                    "mapr": mapr, "operator": operator,
                })
                continue

            scored_parts = score_all_parts(cls_name, labeled_parts, confidence,
                                            exposure, uniqueness, context)
            protected = apply_operator_multiregion(protected, scored_parts)
            per_detection_log.append({
                "class": cls_name, "path": "subregion",
                "parts": [(p[0], p[2], p[3]) for p in scored_parts],
            })
        else:
            mapr = compute_mapr(cls_name, confidence, exposure, uniqueness, context)
            operator = select_operator(mapr)
            protected = apply_operator_obb(protected, corners, operator)
            per_detection_log.append({
                "class": cls_name, "path": "whole_object",
                "mapr": mapr, "operator": operator,
            })

    return protected, per_detection_log


# =============================================================================
# PART O — DIOR + DOTA SUB-REGION EVALUATION (Table 2/3)
# =============================================================================

In [ ]:
def run_dior_subregion_eval(val_dir, sample_size=293, seed=42, conf_threshold=0.4):
    sam_model = get_sam_model()
    random.seed(seed)
    all_images = os.listdir(val_dir)
    sample_images = random.sample(all_images, min(sample_size, len(all_images)))
    print(f"DIOR val set has {len(all_images)} images; evaluating on {len(sample_images)}")

    results_log = {name: {"mae": [], "psnr": [], "ssim": [], "residual_detect": []}
                    for name in ["original", "global_blur", "fixed_mask",
                                 "single_attribute", "amapp_subregion_linear"]}
    subregion_path_count, fallback_count, total_eligible = 0, 0, 0
    skipped_no_detections = 0

    for idx, fname in enumerate(sample_images):
        image_path = os.path.join(val_dir, fname)
        image = cv2.imread(image_path)
        if image is None:
            continue
        img_shape = image.shape

        results = model_dior.predict(source=image_path, save=False,
                                      conf=conf_threshold, verbose=False)
        detections = []
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                cls_id = int(box.cls)
                cls_name = DIOR_CLASSES[cls_id]
                confidence = float(box.conf)
                bbox = box.xyxy[0].tolist()
                detections.append((cls_name, confidence, bbox))

        if not detections:
            skipped_no_detections += 1
            continue

        original_count = len(detections)
        methods = {
            "original": image.copy(),
            "global_blur": baseline_global_blur(image.copy()),
            "fixed_mask": baseline_fixed_mask(image, detections),
            "single_attribute": baseline_single_attribute(image, detections),
        }
        protected, per_detection_log = run_amapp_subregion_linear(image, detections, img_shape, sam_model)
        methods["amapp_subregion_linear"] = protected

        for entry in per_detection_log:
            if entry["class"] in SUBREGION_CLASSES:
                total_eligible += 1
                if entry["path"] == "subregion":
                    subregion_path_count += 1
                elif entry["path"] == "whole_object_fallback_no_masks":
                    fallback_count += 1

        for method_name, protected_img in methods.items():
            if method_name == "original":
                results_log[method_name]["mae"].append(0.0)
                results_log[method_name]["ssim"].append(1.0)
            else:
                mae, psnr_val, ssim_val = compute_utility_metrics(image, protected_img)
                results_log[method_name]["mae"].append(mae)
                results_log[method_name]["psnr"].append(psnr_val)
                results_log[method_name]["ssim"].append(ssim_val)
            residual = compute_residual_detectability(protected_img, model_dior, DIOR_CLASSES, original_count)
            results_log[method_name]["residual_detect"].append(residual)

        if (idx + 1) % 25 == 0:
            print(f"[DIOR] Processed {idx + 1}/{len(sample_images)} images...")
        elif (idx + 1) % 5 == 0:
            print(f"  [DIOR] ...{idx + 1}/{len(sample_images)} "
                  f"(sub-region path used on {subregion_path_count}/{total_eligible} eligible detections)")

    print(f"\n[DIOR] Done. Skipped {skipped_no_detections} images with zero detections.")
    print(f"[DIOR] Sub-region-eligible detections: {total_eligible} total, "
          f"{subregion_path_count} via sub-region path, {fallback_count} fell back to whole-object.")
    return results_log

def run_dota_subregion_eval(val_dir, sample_size=293, seed=42, conf_threshold=0.4):
    sam_model = get_sam_model()
    random.seed(seed)
    all_images = os.listdir(val_dir)
    sample_images = random.sample(all_images, min(sample_size, len(all_images)))
    print(f"DOTA val set has {len(all_images)} images; evaluating on {len(sample_images)}")

    results_log = {name: {"mae": [], "psnr": [], "ssim": [], "residual_detect": []}
                    for name in ["original", "global_blur", "fixed_mask",
                                 "single_attribute", "amapp_subregion_linear"]}
    subregion_path_count, fallback_count, total_eligible = 0, 0, 0
    skipped_no_detections = 0

    for idx, fname in enumerate(sample_images):
        image_path = os.path.join(val_dir, fname)
        image = cv2.imread(image_path)
        if image is None:
            continue
        img_shape = image.shape

        results = model_dota.predict(source=image_path, save=False,
                                      conf=conf_threshold, verbose=False)
        detections = []
        for r in results:
            obb = r.obb
            if obb is None or len(obb) == 0:
                continue
            for i in range(len(obb)):
                cls_id = int(obb.cls[i])
                cls_name = DOTA_CLASSES[cls_id]
                confidence = float(obb.conf[i])
                corners = obb.xyxyxyxy[i].tolist()
                aabb = obb_corners_to_aabb(corners)
                detections.append((cls_name, confidence, aabb, corners))

        if not detections:
            skipped_no_detections += 1
            continue

        original_count = len(detections)
        methods = {
            "original": image.copy(),
            "global_blur": baseline_global_blur(image.copy()),
            "fixed_mask": baseline_fixed_mask_obb(image, detections),
            "single_attribute": baseline_single_attribute_obb(image, detections),
        }
        protected, per_detection_log = run_amapp_subregion_linear_obb(image, detections, img_shape, sam_model)
        methods["amapp_subregion_linear"] = protected

        for entry in per_detection_log:
            if entry["class"] in SUBREGION_CLASSES:
                total_eligible += 1
                if entry["path"] == "subregion":
                    subregion_path_count += 1
                elif entry["path"] == "whole_object_fallback_no_masks":
                    fallback_count += 1

        for method_name, protected_img in methods.items():
            if method_name == "original":
                results_log[method_name]["mae"].append(0.0)
                results_log[method_name]["ssim"].append(1.0)
            else:
                mae, psnr_val, ssim_val = compute_utility_metrics(image, protected_img)
                results_log[method_name]["mae"].append(mae)
                results_log[method_name]["psnr"].append(psnr_val)
                results_log[method_name]["ssim"].append(ssim_val)
            residual = compute_residual_detectability_obb(protected_img, model_dota, original_count)
            results_log[method_name]["residual_detect"].append(residual)

        if (idx + 1) % 25 == 0:
            print(f"[DOTA] Processed {idx + 1}/{len(sample_images)} images...")
        elif (idx + 1) % 5 == 0:
            print(f"  [DOTA] ...{idx + 1}/{len(sample_images)} "
                  f"(sub-region path used on {subregion_path_count}/{total_eligible} eligible detections)")

    print(f"\n[DOTA] Done. Skipped {skipped_no_detections} images with zero detections.")
    print(f"[DOTA] Sub-region-eligible detections: {total_eligible} total, "
          f"{subregion_path_count} via sub-region path, {fallback_count} fell back to whole-object.")
    return results_log

def summarize_and_save(results_log, dataset_label, out_prefix):

    table2_rows, table3_rows = [], []
    for method_name, metrics in results_log.items():
        n = len(metrics["mae"])
        if n == 0:
            continue
        table2_rows.append({
            "Method": method_name,
            "Residual Detectability (mean)": np.mean(metrics["residual_detect"]),
            "Residual Detectability (std)": np.std(metrics["residual_detect"]),
            "Risk Reduction": 1 - np.mean(metrics["residual_detect"]),
            "N images": n,
        })
        psnr_mean = "N/A" if method_name == "original" else np.mean(metrics["psnr"])
        table3_rows.append({
            "Method": method_name,
            "PSNR (mean)": psnr_mean,
            "SSIM (mean)": np.mean(metrics["ssim"]),
            "MAE (mean)": np.mean(metrics["mae"]),
        })

    table2 = pd.DataFrame(table2_rows)
    table3 = pd.DataFrame(table3_rows)

    print(f"\n=== {dataset_label} SUB-REGION (LINEAR) TABLE 2: Privacy Effectiveness ===")
    print(table2.to_string(index=False))
    print(f"\n=== {dataset_label} SUB-REGION (LINEAR) TABLE 3: Utility Preservation ===")
    print(table3.to_string(index=False))

    table2.to_csv(f'/kaggle/working/{out_prefix}_table2_privacy.csv', index=False)
    table3.to_csv(f'/kaggle/working/{out_prefix}_table3_utility.csv', index=False)
    print(f"\nSaved {dataset_label} sub-region (linear) tables.")
    return table2, table3

In [ ]:
def baseline_fixed_mask_obb(image, detections):
    protected = image.copy()
    for cls_name, conf, aabb, corners in detections:
        protected = apply_operator_obb(protected, corners, "mask")
    return protected

def baseline_single_attribute_obb(image, detections):
    protected = image.copy()
    if not detections:
        return protected
    top = max(detections, key=lambda d: d[1])
    protected = apply_operator_obb(protected, top[3], "mask")
    return protected

def compute_utility_metrics(original, protected):
    mae = np.mean(np.abs(original.astype(float) - protected.astype(float)))
    psnr_val = psnr(original, protected, data_range=255)
    ssim_val = ssim(original, protected, channel_axis=2, data_range=255)
    return mae, psnr_val, ssim_val

def compute_residual_detectability_obb(protected_image, model, original_count):
    tmp_path = "/tmp/_tmp_protected_dota.jpg"
    cv2.imwrite(tmp_path, protected_image)
    results = model.predict(source=tmp_path, save=False, conf=0.4, verbose=False)
    residual_count = 0
    for r in results:
        if r.obb is not None:
            residual_count += len(r.obb)
    return residual_count / original_count if original_count > 0 else 0.0

In [ ]:
# =============================================================================
# PART D — BASELINE PROTECTION METHODS
# =============================================================================

def baseline_global_blur(image):
    return cv2.GaussianBlur(image, (51, 51), 0)

def baseline_fixed_mask_obb(image, detections):
    protected = image.copy()
    for cls_name, conf, aabb, corners in detections:
        protected = apply_operator_obb(protected, corners, "mask")
    return protected

def baseline_single_attribute_obb(image, detections):
    protected = image.copy()
    if not detections:
        return protected
    top = max(detections, key=lambda d: d[1])
    protected = apply_operator_obb(protected, top[3], "mask")
    return protected


# =============================================================================
# PART E — WHOLE-OBJECT AMAPP PIPELINE (DIOR + DOTA/OBB)
# =============================================================================

def run_amapp_linear_obb(image, detections, img_shape):
    protected = image.copy()
    scored = []
    for cls_name, confidence, aabb, corners in detections:
        exposure = compute_exposure(aabb, img_shape)
        detections_view = [(d[0], d[1], d[2]) for d in detections]
        uniqueness = compute_uniqueness(cls_name, detections_view)
        context = compute_context(aabb, detections_view)
        mapr = compute_mapr(cls_name, confidence, exposure, uniqueness, context)
        scored.append((cls_name, confidence, aabb, corners, mapr))
    scored.sort(key=lambda d: d[4], reverse=True)
    for cls_name, confidence, aabb, corners, mapr in scored:
        operator = select_operator(mapr)
        protected = apply_operator_obb(protected, corners, operator)
    return protected, scored

def run_amapp_linear(image, detections, img_shape):
    protected = image.copy()
    scored = []
    for cls_name, confidence, bbox in detections:
        exposure = compute_exposure(bbox, img_shape)
        uniqueness = compute_uniqueness(cls_name, detections)
        context = compute_context(bbox, detections)
        mapr = compute_mapr(cls_name, confidence, exposure, uniqueness, context)
        scored.append((cls_name, confidence, bbox, mapr))
    scored.sort(key=lambda d: d[3], reverse=True)
    for cls_name, confidence, bbox, mapr in scored:
        operator = select_operator(mapr)
        protected = apply_operator(protected, bbox, operator)
    return protected, scored


# =============================================================================

In [ ]:
required = [
    "DIOR_CLASSES", "DOTA_CLASSES", "SENSITIVITY_TABLE", "SUBREGION_CLASSES", "SENSITIVITY_TABLE_PARTS",
    "model_dior", "model_dota", "compute_exposure", "compute_uniqueness", "compute_context",
    "compute_mapr", "select_operator", "apply_operator", "apply_operator_obb",
    "apply_operator_to_mask", "apply_operator_multiregion", "baseline_global_blur",
    "baseline_fixed_mask", "baseline_single_attribute", "baseline_fixed_mask_obb",
    "baseline_single_attribute_obb", "get_sam_model", "segment_detections_with_sam_batched",
    "label_parts", "score_all_parts", "run_amapp_subregion_linear", "run_amapp_subregion_linear_obb",
    "compute_utility_metrics", "compute_residual_detectability", "compute_residual_detectability_obb",
    "run_dior_subregion_eval", "run_dota_subregion_eval", "summarize_and_save",
]
missing = [name for name in required if name not in globals()]
print("MISSING:", missing if missing else "none — ready to run")

In [ ]:
# =============================================================================
# PART P — EUROSAT DOWNSTREAM UTILITY TEST (CPU)
# =============================================================================

DEVICE = torch.device("cpu")

CLASS_NAMES = ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
               'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']

def load_eurosat_classifier(checkpoint_path):
    model_cls = models.resnet18(weights=None)
    model_cls.fc = nn.Linear(model_cls.fc.in_features, len(CLASS_NAMES))
    state_dict = torch.load(checkpoint_path, map_location=DEVICE)
    model_cls.load_state_dict(state_dict)
    model_cls = model_cls.to(DEVICE)
    model_cls.eval()
    return model_cls

eurosat_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def classify_image_array(image_bgr, model_cls, transform):
    """Takes a cv2 (BGR, numpy) image, returns predicted class index. CPU only."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(image_rgb)
    tensor = transform(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output = model_cls(tensor)
        _, pred = torch.max(output, 1)
    return int(pred.item())

def run_eurosat_downstream_test(base_dir, model_cls, sample_size=293, seed=42, conf_threshold=0.4):
    """
    base_dir: EuroSAT root folder containing validation.csv and images
              (same layout as used during EuroSAT classifier training).
    model_cls: loaded via load_eurosat_classifier().
    """
    val_df = pd.read_csv(os.path.join(base_dir, "validation.csv"))
    val_sample = val_df.sample(n=min(sample_size, len(val_df)), random_state=seed)

    correct_original, correct_protected, total = 0, 0, 0
    detections_found_count = 0

    for idx, row in val_sample.iterrows():
        img_path = os.path.join(base_dir, row['Filename'])
        if not os.path.exists(img_path):
            # Handle validation.csv formats where Filename already includes
            # the full relative path from base_dir's parent, or an absolute
            # path -- fall back to using row['Filename'] directly rather
            # than silently skipping every image on a path mismatch.
            if os.path.exists(row['Filename']):
                img_path = row['Filename']
        true_label = int(row['Label'])

        image = cv2.imread(img_path)
        if image is None:
            continue
        img_shape = image.shape

        results = model_dior.predict(source=img_path, save=False, conf=conf_threshold, verbose=False)
        detections = []
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                cls_id = int(box.cls)
                cls_name = DIOR_CLASSES[cls_id]
                confidence = float(box.conf)
                xyxy = box.xyxy[0].tolist()
                detections.append((cls_name, confidence, xyxy))

        if detections:
            detections_found_count += 1
            protected_image, _ = run_amapp_linear(image, detections, img_shape)
        else:
            protected_image = image.copy()

        pred_original = classify_image_array(image, model_cls, eurosat_transform)
        pred_protected = classify_image_array(protected_image, model_cls, eurosat_transform)

        if pred_original == true_label:
            correct_original += 1
        if pred_protected == true_label:
            correct_protected += 1
        total += 1

        if total % 25 == 0:
            print(f"Processed {total}/{len(val_sample)} images...")

    if total == 0:
        print(f"\nERROR: 0 of {len(val_sample)} images could be loaded from '{base_dir}'. "
              f"Check that base_dir and validation.csv's Filename column are consistent "
              f"(e.g. row['Filename'] may need to be joined differently with base_dir).")
        return {"total": 0, "detections_found_count": 0, "accuracy_original": None, "accuracy_protected": None}

    print(f"\nEvaluated {total} EuroSAT images")
    print(f"Images with at least 1 DIOR-model detection: {detections_found_count} "
          f"({100*detections_found_count/total:.1f}%)")
    print(f"Classification accuracy, ORIGINAL images: {100*correct_original/total:.2f}%")
    print(f"Classification accuracy, AMAPP (linear)-PROTECTED images: {100*correct_protected/total:.2f}%")
    print(f"Accuracy delta: {100*(correct_protected-correct_original)/total:+.2f} percentage points")

    return {
        "total": total,
        "detections_found_count": detections_found_count,
        "accuracy_original": 100 * correct_original / total,
        "accuracy_protected": 100 * correct_protected / total,
    }

In [ ]:
# =============================================================================
# PART R — TABLE 4 HELPERS (component configs, privacy-utility score)
# =============================================================================

PSNR_NORMALIZATION_CAP = 35.0  # see docstring above for justification

def compute_mapr_ablation(cls_name, confidence, exposure, uniqueness, context, config):
    sensitivity = SENSITIVITY_TABLE.get(cls_name, 0.3)
    if config == 'sensitivity_only':
        return sensitivity
    elif config == 'plus_confidence':
        return 0.5 * sensitivity + 0.5 * confidence
    elif config == 'plus_exposure':
        return 0.4 * sensitivity + 0.3 * confidence + 0.3 * exposure
    elif config == 'full':
        return compute_mapr(cls_name, confidence, exposure, uniqueness, context)
    else:
        raise ValueError(f"Unknown config: {config}")

ABLATION_CONFIGS = ['sensitivity_only', 'plus_confidence', 'plus_exposure', 'full']

CONFIG_DISPLAY_NAMES = {
    'sensitivity_only': 'Sensitivity only',
    'plus_confidence': '+ Confidence',
    'plus_exposure': '+ Exposure',
    'full': 'Full MAPR (linear)',
}

def run_amapp_config(image, detections, img_shape, config):
    protected = image.copy()
    scored = []
    for cls_name, confidence, bbox in detections:
        exposure = compute_exposure(bbox, img_shape)
        uniqueness = compute_uniqueness(cls_name, detections)
        context = compute_context(bbox, detections)
        mapr = compute_mapr_ablation(cls_name, confidence, exposure, uniqueness, context, config)
        scored.append((cls_name, confidence, bbox, mapr))
    scored.sort(key=lambda d: d[3], reverse=True)
    for cls_name, confidence, bbox, mapr in scored:
        operator = select_operator(mapr)
        protected = apply_operator(protected, bbox, operator)
    return protected, scored

def run_table4_component_ablation(val_dir, sample_size=293, seed=42, conf_threshold=0.4):
    random.seed(seed)
    all_images = os.listdir(val_dir)
    sample_images = random.sample(all_images, min(sample_size, len(all_images)))
    print(f"DIOR val set has {len(all_images)} images; evaluating on {len(sample_images)}")

    results_log = {config: {"psnr": [], "residual_detect": []} for config in ABLATION_CONFIGS}
    skipped_no_detections = 0

    for idx, fname in enumerate(sample_images):
        image_path = os.path.join(val_dir, fname)
        image = cv2.imread(image_path)
        if image is None:
            continue
        img_shape = image.shape

        results = model_dior.predict(source=image_path, save=False,
                                      conf=conf_threshold, verbose=False)
        detections = []
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                cls_id = int(box.cls)
                cls_name = DIOR_CLASSES[cls_id]
                confidence = float(box.conf)
                bbox = box.xyxy[0].tolist()
                detections.append((cls_name, confidence, bbox))

        if not detections:
            skipped_no_detections += 1
            continue

        original_count = len(detections)
        for config in ABLATION_CONFIGS:
            protected, _ = run_amapp_config(image, detections, img_shape, config)
            _, psnr_val, _ = compute_utility_metrics(image, protected)
            residual = compute_residual_detectability(protected, model_dior, DIOR_CLASSES, original_count)
            results_log[config]["psnr"].append(psnr_val)
            results_log[config]["residual_detect"].append(residual)

        if (idx + 1) % 25 == 0:
            print(f"Processed {idx + 1}/{len(sample_images)} images...")

    print(f"\nDone. Skipped {skipped_no_detections} images with zero detections.")
    return results_log

def summarize_table4(results_log):

    rows = []
    for config in ABLATION_CONFIGS:
        metrics = results_log[config]
        n = len(metrics["psnr"])
        if n == 0:
            continue
        mean_residual = np.mean(metrics["residual_detect"])
        risk_reduction = 1 - mean_residual
        mean_psnr = np.mean(metrics["psnr"])
        normalized_psnr = min(mean_psnr / PSNR_NORMALIZATION_CAP, 1.0)
        privacy_utility_score = 0.5 * risk_reduction + 0.5 * normalized_psnr

        rows.append({
            "Configuration": CONFIG_DISPLAY_NAMES[config],
            "Risk Reduction": risk_reduction,
            "PSNR (mean)": mean_psnr,
            "Privacy-utility score": privacy_utility_score,
            "N images": n,
        })

    table4 = pd.DataFrame(rows)
    print("\n=== TABLE 4 (MANUSCRIPT): Component Ablation ===")
    print(table4.to_string(index=False))
    table4.to_csv('/kaggle/working/table4_component_ablation.csv', index=False)
    print("\nSaved table4_component_ablation.csv")
    print("\nSANITY CHECK: the 'Full MAPR (linear)' row's Risk Reduction and PSNR")
    print("should closely match this project's existing DIOR Table 2/3 whole-object")
    print("linear results (74.8% risk reduction, PSNR 29.82) -- both use the same")
    print("compute_mapr() formula and the same seed=42 sample. Small differences")
    print("are expected only if the detector or conf_threshold changed since then.")
    return table4


# =============================================================================
# PART S — TABLE 4: A0-A5 (whole-object ablation + weight sensitivity + runtime)
# =============================================================================

def compute_mapr_a0_a4(cls_name, confidence, exposure, uniqueness, context, config,
                        weights=(0.3, 0.25, 0.2, 0.15, 0.1)):
    w_sens, w_conf, w_exp, w_uniq, w_ctx = weights
    sensitivity = SENSITIVITY_TABLE.get(cls_name, 0.3)

    if config == 'A0':
        return w_sens * sensitivity
    elif config == 'A1':
        return w_sens * sensitivity + w_conf * confidence
    elif config == 'A2':
        return w_sens * sensitivity + w_conf * confidence + w_exp * exposure
    elif config == 'A3':
        return (w_sens * sensitivity + w_conf * confidence + w_exp * exposure
                + w_uniq * uniqueness)
    elif config == 'A4':
        return compute_mapr(cls_name, confidence, exposure, uniqueness, context, weights)
    else:
        raise ValueError(f"Unknown A0-A4 config: {config}")

A5_WEIGHT_PERTURBATIONS = {
    'A5_baseline':        (0.30, 0.25, 0.20, 0.15, 0.10),  # = A4, included for direct comparison
    'A5_sens_up':         (0.45, 0.20, 0.15, 0.12, 0.08),  # sensitivity weighted more heavily
    'A5_sens_down':       (0.15, 0.30, 0.25, 0.18, 0.12),  # sensitivity weighted less heavily
    'A5_conf_up':         (0.25, 0.40, 0.15, 0.12, 0.08),  # confidence weighted more heavily
    'A5_equal':           (0.20, 0.20, 0.20, 0.20, 0.20),  # all five components equal
}

A0_A4_CONFIGS = ['A0', 'A1', 'A2', 'A3', 'A4']

A0_A4_DISPLAY_NAMES = {
    'A0': 'A0: Sensitivity only',
    'A1': 'A1: + Confidence',
    'A2': 'A2: + Exposure',
    'A3': 'A3: + Uniqueness',
    'A4': 'A4: Full MAPR (+ Context)',
}

def run_amapp_a0_a4(image, detections, img_shape, config):
    protected = image.copy()
    scored = []
    for cls_name, confidence, bbox in detections:
        exposure = compute_exposure(bbox, img_shape)
        uniqueness = compute_uniqueness(cls_name, detections)
        context = compute_context(bbox, detections)
        mapr = compute_mapr_a0_a4(cls_name, confidence, exposure, uniqueness, context, config)
        scored.append((cls_name, confidence, bbox, mapr))
    scored.sort(key=lambda d: d[3], reverse=True)
    for cls_name, confidence, bbox, mapr in scored:
        operator = select_operator(mapr)
        protected = apply_operator(protected, bbox, operator)
    return protected, scored

def run_amapp_a5(image, detections, img_shape, weight_vector):
    protected = image.copy()
    scored = []
    for cls_name, confidence, bbox in detections:
        exposure = compute_exposure(bbox, img_shape)
        uniqueness = compute_uniqueness(cls_name, detections)
        context = compute_context(bbox, detections)
        mapr = compute_mapr(cls_name, confidence, exposure, uniqueness, context, weight_vector)
        scored.append((cls_name, confidence, bbox, mapr))
    scored.sort(key=lambda d: d[3], reverse=True)
    for cls_name, confidence, bbox, mapr in scored:
        operator = select_operator(mapr)
        protected = apply_operator(protected, bbox, operator)
    return protected, scored

def run_table4_full_ablation(val_dir, sample_size=293, seed=42, conf_threshold=0.4):
    random.seed(seed)
    all_images = os.listdir(val_dir)
    sample_images = random.sample(all_images, min(sample_size, len(all_images)))
    print(f"DIOR val set has {len(all_images)} images; evaluating on {len(sample_images)}")

    all_configs = A0_A4_CONFIGS + list(A5_WEIGHT_PERTURBATIONS.keys())
    results_log = {config: {"psnr": [], "residual_detect": [], "runtime_sec": []}
                    for config in all_configs}
    skipped_no_detections = 0

    for idx, fname in enumerate(sample_images):
        image_path = os.path.join(val_dir, fname)
        image = cv2.imread(image_path)
        if image is None:
            continue
        img_shape = image.shape

        results = model_dior.predict(source=image_path, save=False,
                                      conf=conf_threshold, verbose=False)
        detections = []
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                cls_id = int(box.cls)
                cls_name = DIOR_CLASSES[cls_id]
                confidence = float(box.conf)
                bbox = box.xyxy[0].tolist()
                detections.append((cls_name, confidence, bbox))

        if not detections:
            skipped_no_detections += 1
            continue

        original_count = len(detections)

        for config in A0_A4_CONFIGS:
            t0 = time.perf_counter()
            protected, _ = run_amapp_a0_a4(image, detections, img_shape, config)
            runtime = time.perf_counter() - t0
            _, psnr_val, _ = compute_utility_metrics(image, protected)
            residual = compute_residual_detectability(protected, model_dior, DIOR_CLASSES, original_count)
            results_log[config]["psnr"].append(psnr_val)
            results_log[config]["residual_detect"].append(residual)
            results_log[config]["runtime_sec"].append(runtime)

        for name, weight_vector in A5_WEIGHT_PERTURBATIONS.items():
            t0 = time.perf_counter()
            protected, _ = run_amapp_a5(image, detections, img_shape, weight_vector)
            runtime = time.perf_counter() - t0
            _, psnr_val, _ = compute_utility_metrics(image, protected)
            residual = compute_residual_detectability(protected, model_dior, DIOR_CLASSES, original_count)
            results_log[name]["psnr"].append(psnr_val)
            results_log[name]["residual_detect"].append(residual)
            results_log[name]["runtime_sec"].append(runtime)

        if (idx + 1) % 25 == 0:
            print(f"Processed {idx + 1}/{len(sample_images)} images...")

    print(f"\nDone. Skipped {skipped_no_detections} images with zero detections.")
    return results_log

def summarize_table4_full(results_log):

    def _row(display_name, metrics):
        n = len(metrics["psnr"])
        if n == 0:
            return None
        mean_residual = np.mean(metrics["residual_detect"])
        risk_reduction = 1 - mean_residual
        mean_psnr = np.mean(metrics["psnr"])
        normalized_psnr = min(mean_psnr / PSNR_NORMALIZATION_CAP, 1.0) if np.isfinite(mean_psnr) else 1.0
        privacy_utility_score = 0.5 * risk_reduction + 0.5 * normalized_psnr
        mean_runtime_ms = np.mean(metrics["runtime_sec"]) * 1000
        return {
            "Configuration": display_name,
            "Risk Reduction": risk_reduction,
            "PSNR (mean)": mean_psnr,
            "Privacy-utility score": privacy_utility_score,
            "Runtime (ms/image, mean)": mean_runtime_ms,
            "N images": n,
        }

    a0_a4_rows = [_row(A0_A4_DISPLAY_NAMES[c], results_log[c]) for c in A0_A4_CONFIGS]
    a0_a4_rows = [r for r in a0_a4_rows if r is not None]
    table4_a0a4 = pd.DataFrame(a0_a4_rows)

    a5_rows = [_row(name, results_log[name]) for name in A5_WEIGHT_PERTURBATIONS]
    a5_rows = [r for r in a5_rows if r is not None]
    table4_a5 = pd.DataFrame(a5_rows)

    print("\n=== TABLE 4a (A0-A4): Component Ablation ===")
    print(table4_a0a4.to_string(index=False))
    print("\n=== TABLE 4b (A5): Weight/Threshold Sensitivity, Full MAPR ===")
    print(table4_a5.to_string(index=False))

    table4_a0a4.to_csv('/kaggle/working/table4a_component_ablation_full.csv', index=False)
    table4_a5.to_csv('/kaggle/working/table4b_weight_sensitivity.csv', index=False)
    print("\nSaved table4a_component_ablation_full.csv and table4b_weight_sensitivity.csv")

    print("\nSANITY CHECKS:")
    print("- A4's Risk Reduction/PSNR should be close to this project's existing")
    print("  DIOR Table 2/3 whole-object linear results (74.8%, PSNR 29.82).")
    print("- A5_baseline uses the SAME weights as A4 -- its numbers should be")
    print("  IDENTICAL to A4's (this is a built-in consistency check, not an")
    print("  independent finding -- if they differ noticeably, something is wrong).")
    return table4_a0a4, table4_a5

In [ ]:
CFG = dict(
    DIOR_VAL_DIR    = "/kaggle/input/datasets/kefanzhan/diordataset/Dior/Dior/images/val",
    DOTA_VAL_DIR    = "/tmp/data/DOTAv1/images/val",   # <<< fill in
    EUROSAT_DIR     = "/kaggle/input/datasets/apollo2506/eurosat-dataset/EuroSAT",   # <<< folder containing validation.csv
    EUROSAT_WEIGHTS = "/kaggle/input/notebooks/sowshan1/amapp-eurosat/eurosat_checkpoints/best.pt",
    OUT = "/kaggle/working",
    SAMPLE = 293, SEED = 42, CONF = 0.4, ATTACK_CONF = 0.25,
    BUDGET_MIN = dict(dior=35, eurosat=8, dota=35),
    RUN = dict(dior=False, ablation=False, eurosat=False, dota=False),
    N_BOOT = 2000,
)

In [ ]:
# ###########################################################################
# CELL C1 — PREFLIGHT, LOGGING, ATTACKER HELPERS, UNIFIED PER-IMAGE LOOP
# (needs CFG from the config cell, and every earlier cell already run)
# ###########################################################################
import time, traceback, json, warnings
import numpy as np, pandas as pd, cv2, os, random

T0 = time.time()
OUT = CFG["OUT"]
os.makedirs(OUT, exist_ok=True)
LOG_PATH = os.path.join(OUT, "run_log.txt")

def log(msg):
    line = f"[{(time.time() - T0) / 60:6.1f} min] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a") as f:
        f.write(line + "\n")

# ---- shared-class map: lets a detector trained on a different dataset act as attacker
SHARED = {
    "airplane": "plane", "plane": "plane",
    "ship": "ship", "boat": "ship",
    "storagetank": "storagetank", "storage-tank": "storagetank",
    "baseballfield": "baseball", "baseball-diamond": "baseball",
    "tenniscourt": "tennis", "tennis-court": "tennis",
    "basketballcourt": "basketball", "basketball-court": "basketball",
    "groundtrackfield": "groundtrack", "ground-track-field": "groundtrack",
    "harbor": "harbor", "bridge": "bridge",
    "vehicle": "vehicle", "large-vehicle": "vehicle", "small-vehicle": "vehicle",
    "car": "vehicle", "truck": "vehicle", "bus": "vehicle",
}
_TMP_ATK = os.path.join(OUT, "_tmp_attack.jpg")

def attacker_count(model, image_bgr, kind):
    """Detections of shared classes by an attacker model. The image goes through the same
    JPEG write/read as the primary residual-detectability metric, so the two are comparable."""
    cv2.imwrite(_TMP_ATK, image_bgr)
    res = model.predict(source=_TMP_ATK, save=False, conf=CFG["ATTACK_CONF"], verbose=False)
    n = 0
    for r in res:
        det = r.obb if kind == "obb" else r.boxes
        if det is None or len(det) == 0:
            continue
        for c in det.cls:
            if model.names.get(int(c), "") in SHARED:
                n += 1
    return n

def fidelity_extras(orig, prot):
    """Fraction of pixels changed, and PSNR computed only over the changed pixels."""
    diff = np.any(orig != prot, axis=2)
    frac = float(diff.mean())
    if frac == 0.0:
        return 0.0, np.nan
    err = (orig.astype(np.float32) - prot.astype(np.float32)) ** 2
    mse = float(err[diff].mean())
    return frac, (10 * np.log10(255.0 ** 2 / mse) if mse > 0 else np.inf)

# ---- record the environment (the manuscript states Ultralytics 8.4.139 / PyTorch 2.10.0+cu128)
try:
    import ultralytics, torch
    log(f"ENV: ultralytics {ultralytics.__version__}, torch {torch.__version__}, "
        f"cuda available: {torch.cuda.is_available()}")
except Exception as e:
    log(f"could not read environment versions: {e}")

# ---- preflight: fail early and clearly instead of 30 minutes in
SAM_MODEL, COCO = None, None
try:
    SAM_MODEL = get_sam_model()
    log("SAM loaded OK")
except Exception as e:
    log(f"!! SAM FAILED to load ({e}). Is Internet switched ON in notebook settings? "
        f"DIOR/DOTA sub-region stages will be skipped.")
try:
    COCO = YOLO("yolo11n.pt")
    log("COCO attacker (yolo11n) loaded OK")
except Exception as e:
    log(f"COCO attacker unavailable ({e}); continuing without it")

for key, stage in [("DIOR_VAL_DIR", "dior"), ("DIOR_VAL_DIR", "ablation"),
                   ("DOTA_VAL_DIR", "dota"), ("EUROSAT_DIR", "eurosat")]:
    if CFG["RUN"][stage] and not os.path.isdir(CFG[key]):
        log(f"!! {key} does not exist: {CFG[key]}  -> stage '{stage}' will fail")

def run_unified(tag, val_dir, extract, build, residual_fn, attackers, budget_min, out_csv):
    """Same sampling/skip logic as run_dior_subregion_eval / run_dota_subregion_eval, but every
    method is scored on the SAME images in one pass and every per-image value is kept."""
    random.seed(CFG["SEED"])
    all_images = os.listdir(val_dir)
    sample = random.sample(all_images, min(CFG["SAMPLE"], len(all_images)))
    log(f"[{tag}] {len(all_images)} images in folder; sampled {len(sample)} (seed {CFG['SEED']})")
    rows, failures = [], []
    skipped_nodet, skipped_unreadable, done = 0, 0, 0
    t_stage = time.time()
    for idx, fname in enumerate(sample):
        if (time.time() - t_stage) / 60 > budget_min:
            log(f"[{tag}] TIME BUDGET ({budget_min} min) reached after {done} images; stopping stage")
            break
        try:
            path = os.path.join(val_dir, fname)
            image = cv2.imread(path)
            if image is None:
                skipped_unreadable += 1
                continue
            dets = extract(path)
            if not dets:
                skipped_nodet += 1
                continue
            n0 = len(dets)
            methods, rts, stats = build(image, dets, image.shape)
            img_rows = []
            for mname, prot in methods.items():
                row = {"image": fname, "img_idx": done, "method": mname, "n_dets": n0}
                if mname == "original":
                    row.update(mae=0.0, psnr=np.nan, ssim=1.0)
                    frac, psnr_r = 0.0, np.nan
                else:
                    mae, psnr_v, ssim_v = compute_utility_metrics(image, prot)
                    row.update(mae=mae, psnr=psnr_v, ssim=ssim_v)
                    frac, psnr_r = fidelity_extras(image, prot)
                row["residual"] = residual_fn(prot, n0)
                row["changed_frac"], row["psnr_changed_region"] = frac, psnr_r
                for aname, (model, kind) in attackers.items():
                    row[f"atk_{aname}"] = attacker_count(model, prot, kind)
                row["runtime_ms"] = rts.get(mname, np.nan)
                if mname == "amapp_subregion_linear":
                    row.update(stats)
                img_rows.append(row)
            rows.extend(img_rows)     # only after the whole image succeeded -> methods stay aligned
            done += 1
        except Exception:
            failures.append({"image": fname, "trace": traceback.format_exc()})
            log(f"[{tag}] image {fname} FAILED (see failures json); continuing")
            try:
                import torch; torch.cuda.empty_cache()
            except Exception:
                pass
        if (idx + 1) % 25 == 0:
            pd.DataFrame(rows).to_csv(out_csv, index=False)
            log(f"[{tag}] {idx + 1}/{len(sample)} visited, {done} scored")
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    with open(out_csv.replace(".csv", "_failures.json"), "w") as f:
        json.dump(failures, f, indent=1)
    log(f"[{tag}] DONE: scored {done}, skipped(no detections) {skipped_nodet}, "
        f"unreadable {skipped_unreadable}, failed {len(failures)} -> {out_csv}")
    return pd.DataFrame(rows)


# ###########################################################################
# CELL C2 — DIOR: all methods + whole-object vs sub-region + attackers
# ###########################################################################
def dior_extract(image_path):
    results = model_dior.predict(source=image_path, save=False, conf=CFG["CONF"], verbose=False)
    dets = []
    for r in results:
        if r.boxes is None:
            continue
        for box in r.boxes:
            dets.append((DIOR_CLASSES[int(box.cls)], float(box.conf), box.xyxy[0].tolist()))
    return dets

def dior_build(image, dets, shape):
    m = {"original": image.copy(),
         "global_blur": baseline_global_blur(image.copy()),
         "fixed_mask": baseline_fixed_mask(image, dets),
         "single_attribute": baseline_single_attribute(image, dets)}
    rts = {}
    t = time.perf_counter()
    m["amapp_whole_linear"], _ = run_amapp_linear(image, dets, shape)
    rts["amapp_whole_linear"] = (time.perf_counter() - t) * 1000
    t = time.perf_counter()
    m["amapp_subregion_linear"], plog = run_amapp_subregion_linear(image, dets, shape, SAM_MODEL)
    rts["amapp_subregion_linear"] = (time.perf_counter() - t) * 1000
    elig = [e for e in plog if e["class"] in SUBREGION_CLASSES]
    stats = {"n_eligible": len(elig),
             "n_subregion": sum(e["path"] == "subregion" for e in elig),
             "n_fallback": sum(e["path"] == "whole_object_fallback_no_masks" for e in elig)}
    return m, rts, stats

df_dior = None
if CFG["RUN"]["dior"] and SAM_MODEL is not None:
    try:
        atk = {"dota_xfer": (model_dota, "obb")}
        if COCO is not None:
            atk["coco"] = (COCO, "box")
        df_dior = run_unified(
            "DIOR", CFG["DIOR_VAL_DIR"], dior_extract, dior_build,
            lambda p, n: compute_residual_detectability(p, model_dior, DIOR_CLASSES, n),
            atk, CFG["BUDGET_MIN"]["dior"], os.path.join(OUT, "PERIMAGE_dior.csv"))
    except Exception:
        log("DIOR stage crashed:\n" + traceback.format_exc())


# ###########################################################################
# CELL C3 — Table 4a/4b ablation (DIOR, whole-object): per-image dump + original tables
# ###########################################################################
df_abl = None
if CFG["RUN"]["ablation"]:
    try:
        abl = run_table4_full_ablation(CFG["DIOR_VAL_DIR"], CFG["SAMPLE"], CFG["SEED"], CFG["CONF"])
        rows = []
        for cname, m in abl.items():
            for i in range(len(m["psnr"])):
                rows.append({"config": cname, "img_idx": i, "psnr": m["psnr"][i],
                             "residual": m["residual_detect"][i],
                             "runtime_ms": m["runtime_sec"][i] * 1000})
        df_abl = pd.DataFrame(rows)
        df_abl.to_csv(os.path.join(OUT, "PERIMAGE_ablation.csv"), index=False)
        log(f"[ablation] saved {len(df_abl)} rows")
        try:
            summarize_table4_full(abl)      # original printout + its own CSVs
        except Exception as e:
            log(f"[ablation] original summary printer failed ({e}); per-image CSV is already saved")
    except Exception:
        log("Ablation stage crashed:\n" + traceback.format_exc())


# ###########################################################################
# CELL C4 — EuroSAT: per-image, with the detected-subset accuracy (needs Part P cell)
# ###########################################################################
df_euro = None
if CFG["RUN"]["eurosat"]:
    try:
        model_cls = load_eurosat_classifier(CFG["EUROSAT_WEIGHTS"])
        val_df = pd.read_csv(os.path.join(CFG["EUROSAT_DIR"], "validation.csv"))
        samp = val_df.sample(n=min(CFG["SAMPLE"], len(val_df)), random_state=CFG["SEED"])
        rows, t_stage = [], time.time()
        for _, r in samp.iterrows():
            if (time.time() - t_stage) / 60 > CFG["BUDGET_MIN"]["eurosat"]:
                log("[eurosat] time budget reached"); break
            p = os.path.join(CFG["EUROSAT_DIR"], r["Filename"])
            if not os.path.exists(p) and os.path.exists(r["Filename"]):
                p = r["Filename"]
            image = cv2.imread(p)
            if image is None:
                continue
            res = model_dior.predict(source=p, save=False, conf=CFG["CONF"], verbose=False)
            dets = []
            for rr in res:
                if rr.boxes is None:
                    continue
                for box in rr.boxes:
                    dets.append((DIOR_CLASSES[int(box.cls)], float(box.conf), box.xyxy[0].tolist()))
            prot = run_amapp_linear(image, dets, image.shape)[0] if dets else image.copy()
            true = int(r["Label"])
            rows.append({"file": r["Filename"], "n_dets": len(dets), "detected": int(len(dets) > 0),
                         "correct_orig": int(classify_image_array(image, model_cls, eurosat_transform) == true),
                         "correct_prot": int(classify_image_array(prot, model_cls, eurosat_transform) == true)})
        df_euro = pd.DataFrame(rows)
        df_euro.to_csv(os.path.join(OUT, "PERIMAGE_eurosat.csv"), index=False)
        log(f"[eurosat] scored {len(df_euro)} images; {int(df_euro.detected.sum())} had a detection")
    except Exception:
        log("EuroSAT stage crashed:\n" + traceback.format_exc())


# ###########################################################################
# CELL C5 — DOTA: whole-object (linear, never run before) + sub-region + attackers
# ###########################################################################
def dota_extract(image_path):
    results = model_dota.predict(source=image_path, save=False, conf=CFG["CONF"], verbose=False)
    dets = []
    for r in results:
        obb = r.obb
        if obb is None or len(obb) == 0:
            continue
        for i in range(len(obb)):
            corners = obb.xyxyxyxy[i].tolist()
            dets.append((DOTA_CLASSES[int(obb.cls[i])], float(obb.conf[i]),
                         obb_corners_to_aabb(corners), corners))
    return dets

def dota_build(image, dets, shape):
    m = {"original": image.copy(),
         "global_blur": baseline_global_blur(image.copy()),
         "fixed_mask": baseline_fixed_mask_obb(image, dets),
         "single_attribute": baseline_single_attribute_obb(image, dets)}
    rts = {}
    t = time.perf_counter()
    m["amapp_whole_linear"], _ = run_amapp_linear_obb(image, dets, shape)
    rts["amapp_whole_linear"] = (time.perf_counter() - t) * 1000
    t = time.perf_counter()
    m["amapp_subregion_linear"], plog = run_amapp_subregion_linear_obb(image, dets, shape, SAM_MODEL)
    rts["amapp_subregion_linear"] = (time.perf_counter() - t) * 1000
    elig = [e for e in plog if e["class"] in SUBREGION_CLASSES]
    stats = {"n_eligible": len(elig),
             "n_subregion": sum(e["path"] == "subregion" for e in elig),
             "n_fallback": sum(e["path"] == "whole_object_fallback_no_masks" for e in elig)}
    return m, rts, stats

df_dota = None
if CFG["RUN"]["dota"] and SAM_MODEL is not None:
    try:
        atk = {"dior_xfer": (model_dior, "box")}
        if COCO is not None:
            atk["coco"] = (COCO, "box")
        df_dota = run_unified(
            "DOTA", CFG["DOTA_VAL_DIR"], dota_extract, dota_build,
            lambda p, n: compute_residual_detectability_obb(p, model_dota, n),
            atk, CFG["BUDGET_MIN"]["dota"], os.path.join(OUT, "PERIMAGE_dota.csv"))
    except Exception:
        log("DOTA stage crashed:\n" + traceback.format_exc())


# ###########################################################################
# CELL C6 — BOOTSTRAP ANALYSIS + SELF-CHECK (reads the saved CSVs, CPU only)
# ###########################################################################
B = CFG["N_BOOT"]
rng = np.random.default_rng(CFG["SEED"])
S = []                                   # summary lines -> printed and saved
def out(line=""):
    print(line); S.append(line)

def bsmean(v, idx):
    """Bootstrap means of v over resample index matrix idx; non-finite values are ignored."""
    v = np.asarray(v, float)
    v = np.where(np.isfinite(v), v, np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return np.nanmean(v[idx], axis=1)

def ci(a):
    lo, hi = np.nanpercentile(a, [2.5, 97.5])
    return lo, hi

def pu_from(rr, psnr):
    return 0.5 * rr + 0.5 * np.minimum(psnr / PSNR_NORMALIZATION_CAP, 1.0)

REF = {  # numbers already in your results doc: (risk reduction %, PSNR)
    "DIOR": {"global_blur": (75.1, 21.94), "fixed_mask": (93.4, 18.46),
             "single_attribute": (43.5, 22.99), "amapp_subregion_linear": (67.9, 30.31),
             "amapp_whole_linear": (74.8, 29.82)},
    "DOTA": {"global_blur": (39.2, 22.91), "fixed_mask": (80.8, 22.28),
             "single_attribute": (12.7, 33.32), "amapp_subregion_linear": (47.2, 33.08)},
}
PAIRS = [("amapp_subregion_linear", "fixed_mask"), ("amapp_subregion_linear", "global_blur"),
         ("amapp_subregion_linear", "single_attribute"), ("amapp_subregion_linear", "amapp_whole_linear")]

def analyze_main(df, label):
    out("=" * 100); out(f"{label}: main results with 95% bootstrap CIs  (B={B})"); out("=" * 100)
    W = {c: df.pivot(index="img_idx", columns="method", values=c)
         for c in ["residual", "psnr", "ssim", "mae", "changed_frac", "psnr_changed_region"]}
    n = len(W["residual"])
    idx = rng.integers(0, n, size=(B, n))       # ONE resample matrix -> everything is paired
    methods = list(W["residual"].columns)
    rr_bs, ps_bs, rr_pt, ps_pt, rows = {}, {}, {}, {}, []
    for m in methods:
        rr_bs[m] = 1 - bsmean(W["residual"][m], idx)
        rr_pt[m] = 1 - W["residual"][m].mean()
        fin = W["psnr"][m].replace([np.inf, -np.inf], np.nan)
        ps_bs[m] = bsmean(W["psnr"][m], idx)
        ps_pt[m] = fin.mean()
        n_inf = int(np.isinf(W["psnr"][m]).sum())
        rlo, rhi = ci(rr_bs[m]); plo, phi = ci(ps_bs[m]) if m != "original" else (np.nan, np.nan)
        pu_pt = pu_from(rr_pt[m], ps_pt[m]) if m != "original" else np.nan
        rows.append({"method": m, "N": n,
                     "RR_%": 100 * rr_pt[m], "RR_lo": 100 * rlo, "RR_hi": 100 * rhi,
                     "PSNR": ps_pt[m], "PSNR_lo": plo, "PSNR_hi": phi, "n_inf_psnr": n_inf,
                     "SSIM": W["ssim"][m].mean(), "MAE": W["mae"][m].mean(),
                     "changed_px_%": 100 * W["changed_frac"][m].mean(),
                     "PSNR_changed_region": W["psnr_changed_region"][m].replace([np.inf, -np.inf], np.nan).mean(),
                     "PU": pu_pt})
    T = pd.DataFrame(rows)
    out(T.round(3).to_string(index=False))
    T.to_csv(os.path.join(OUT, f"CI_main_{label}.csv"), index=False)

    out(f"\n-- paired differences (A minus B), same {n} images in every method --")
    for a, b in PAIRS:
        if a in methods and b in methods:
            d_rr = 100 * (rr_bs[a] - rr_bs[b]); d_ps = ps_bs[a] - ps_bs[b]
            l1, h1 = ci(d_rr); l2, h2 = ci(d_ps)
            out(f"{a} - {b}:  dRR = {100 * (rr_pt[a] - rr_pt[b]):+.1f} pts [{l1:+.1f}, {h1:+.1f}]"
                f"   dPSNR = {ps_pt[a] - ps_pt[b]:+.2f} dB [{l2:+.2f}, {h2:+.2f}]")

    out("\n-- self-check vs numbers already in your results doc (RR %, PSNR) --")
    for m, (rr_ref, ps_ref) in REF[label].items():
        if m in methods:
            flag = "" if (abs(100 * rr_pt[m] - rr_ref) <= 3 and abs(ps_pt[m] - ps_ref) <= 1.0) else "   <-- MISMATCH"
            out(f"{m:26s} measured {100 * rr_pt[m]:5.1f}% / {ps_pt[m]:5.2f} dB   reference {rr_ref:5.1f}% / {ps_ref:5.2f} dB{flag}")

    if "n_eligible" in df.columns:
        s = df[df.method == "amapp_subregion_linear"]
        te, ts, tf = s.n_eligible.sum(), s.n_subregion.sum(), s.n_fallback.sum()
        out(f"\nsub-region eligible detections: {int(te)} total, {int(ts)} via sub-region path, {int(tf)} fell back to whole-object")
    rt = df.groupby("method").runtime_ms.mean().dropna()
    out("mean runtime per image (ms): " + ", ".join(f"{k}={v:.0f}" for k, v in rt.items()))

    for ac in [c for c in df.columns if c.startswith("atk_")]:
        Wa = df.pivot(index="img_idx", columns="method", values=ac)
        valid = Wa["original"] > 0
        nv = int(valid.sum())
        out(f"\n-- independent attacker '{ac[4:]}': images where it finds >=1 shared-class object on the unprotected image: {nv}/{n} --")
        if nv < 20:
            out("   too few valid images for a meaningful estimate; skipped"); continue
        Wv = Wa[valid]
        idx2 = rng.integers(0, nv, size=(B, nv))
        orig_cnt = Wv["original"].to_numpy(float)
        arows = []
        for m in Wv.columns:
            if m == "original":
                continue
            ratio = Wv[m].to_numpy(float) / orig_cnt
            bs = 1 - bsmean(ratio, idx2); lo, hi = ci(bs)
            arows.append({"method": m, "attacker_RR_%": 100 * (1 - ratio.mean()), "lo": 100 * lo, "hi": 100 * hi,
                          "primary_RR_%_same_images": 100 * (1 - W["residual"].loc[valid, m].mean())})
        out(pd.DataFrame(arows).round(1).to_string(index=False))
    out("")
    return T

if df_dior is not None and len(df_dior):
    analyze_main(df_dior, "DIOR")
if df_dota is not None and len(df_dota):
    analyze_main(df_dota, "DOTA")

if df_abl is not None and len(df_abl):
    out("=" * 100); out("Table 4a/4b ablation (DIOR whole-object) with 95% bootstrap CIs"); out("=" * 100)
    Wr = df_abl.pivot(index="img_idx", columns="config", values="residual")
    Wp = df_abl.pivot(index="img_idx", columns="config", values="psnr")
    n = len(Wr); idx = rng.integers(0, n, size=(B, n)); rows = []
    rr_bs, pu_bs = {}, {}
    for c in Wr.columns:
        rr_bs[c] = 1 - bsmean(Wr[c], idx)
        n_inf = int(np.isinf(Wp[c]).sum())
        ps_b = bsmean(Wp[c], idx); ps_pt = Wp[c].replace([np.inf, -np.inf], np.nan).mean()
        rr_pt = 1 - Wr[c].mean()
        pu_ok = n_inf == 0
        if pu_ok:
            pu_bs[c] = pu_from(rr_bs[c], ps_b)
        rl, rh = ci(rr_bs[c]); pl, ph = ci(ps_b)
        rows.append({"config": c, "N": n, "RR_%": 100 * rr_pt, "RR_lo": 100 * rl, "RR_hi": 100 * rh,
                     "PSNR": ps_pt, "PSNR_lo": pl, "PSNR_hi": ph, "n_inf_psnr": n_inf,
                     "PU": pu_from(rr_pt, ps_pt) if pu_ok else np.nan,
                     "PU_lo": ci(pu_bs[c])[0] if pu_ok else np.nan, "PU_hi": ci(pu_bs[c])[1] if pu_ok else np.nan})
    T4 = pd.DataFrame(rows); out(T4.round(3).to_string(index=False))
    T4.to_csv(os.path.join(OUT, "CI_ablation.csv"), index=False)
    out("(PU left blank where any image has infinite PSNR -- no protection applied -- since PSNR is undefined there)")
    for a, b in [("A5_conf_up", "A5_baseline"), ("A5_sens_down", "A5_baseline")]:
        if a in pu_bs and b in pu_bs:
            d = pu_bs[a] - pu_bs[b]; lo, hi = ci(d)
            out(f"PU({a}) - PU({b}) = {np.nanmean(d):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]"
                f"  -> {'CI excludes 0' if (lo > 0 or hi < 0) else 'CI includes 0 (difference not distinguishable from noise)'}")
    out("")

if df_euro is not None and len(df_euro):
    out("=" * 100); out("EuroSAT (whole-object AMAPP, as in the existing Table 3) with 95% CIs"); out("=" * 100)
    n = len(df_euro); idx = rng.integers(0, n, size=(B, n))
    det = df_euro.detected.to_numpy(float)
    co, cp = df_euro.correct_orig.to_numpy(float), df_euro.correct_prot.to_numpy(float)
    def line(name, v):
        lo, hi = ci(bsmean(v, idx)); return f"{name}: {100 * np.mean(v):.2f}%  [{100 * lo:.2f}, {100 * hi:.2f}]"
    out(f"N = {n}"); out(line("fraction of images with >=1 detection (i.e. actually protected)", det))
    out(line("accuracy, original, all images", co)); out(line("accuracy, protected, all images", cp))
    m = det == 1
    if m.sum() >= 5:
        # ratio-type statistic on the detected subset: resample images, keep only detected ones
        bo, bp = [], []
        for row in idx:
            sel = row[m[row]]
            bo.append(co[sel].mean()); bp.append(cp[sel].mean())
        bo, bp = np.array(bo), np.array(bp)
        out(f"detected subset (n={int(m.sum())}): accuracy original {100 * co[m].mean():.2f}% -> protected {100 * cp[m].mean():.2f}%"
            f"   delta {100 * (cp[m].mean() - co[m].mean()):+.2f} pts  95% CI [{100 * ci(bp - bo)[0]:+.2f}, {100 * ci(bp - bo)[1]:+.2f}]")
    out(f"undetected subset (n={int((~m).sum())}): accuracy original {100 * co[~m].mean():.2f}% vs protected {100 * cp[~m].mean():.2f}%  (identical images -> must match)")
    out("")

with open(os.path.join(OUT, "RESULTS_SUMMARY.txt"), "w") as f:
    f.write("\n".join(S))
log("ALL DONE. Paste RESULTS_SUMMARY.txt (and run_log.txt) back to Claude.")

In [ ]:
# ###########################################################################
# CELL H — ONE-CELL FINAL RUN  (replaces cells A, B, CFG, C, E, F, G and the smoke tests)
# Needs only: cells 0-11 of the notebook + the bridge cell, run before this one.
# Does:  (1) DIOR adaptivity tests on the pinned 286-image sample  (~7 min)
#        (2) DOTA rerun, memory-safe, resumable                    (<= DOTA_BUDGET_MIN)
#        (3) bootstrap analysis + one zip of everything to send back
# Edit ONLY the DOTA_VAL_DIR path (and OBB_MASK_MODE if your repo differs).
# ###########################################################################
import os, gc, time, json, random, shutil, zipfile, traceback, warnings
import numpy as np, pandas as pd, cv2
from concurrent.futures import ThreadPoolExecutor

CFG = dict(
    DIOR_VAL_DIR = "/kaggle/input/datasets/kefanzhan/diordataset/Dior/Dior/images/val",
    DOTA_VAL_DIR    = "/tmp/data/DOTAv1/images/val",   # <<< fill in
    OUT = "/kaggle/working",
    SAMPLE = 293, SEED = 42, CONF = 0.4, ATTACK_CONF = 0.25, N_BOOT = 2000,
)
DOTA_BUDGET_MIN = 80          # stage stops here and still saves/analyses what it has
DOTA_MAX_MP = 40              # images above this many megapixels are skipped and listed (memory safety)
OBB_MASK_MODE = "polygon"     # "polygon": protect the oriented polygon | "aabb": protect its bounding rectangle. Must match your repo's apply_operator_obb.
INCLUDE_STORAGE_TANK_SUBREGION = False   # DOTA spells it "storage-tank"; False keeps sub-region eligibility as in the first run (harbor only)
RUN_F, RUN_DOTA, SKIP_F_IF_DONE = True, True, True
F_BUDGET_MIN = 30
PSNR_NORMALIZATION_CAP = 35.0

# the 286 DIOR images behind Tables 4-9 (os.listdir order differs between sessions, so the sample is stored explicitly)
DIOR_SAMPLE = [
    '18846.jpg', '21113.jpg', '16500.jpg', '18594.jpg', '22702.jpg', '18012.jpg',
    '18713.jpg', '13199.jpg', '19370.jpg', '22849.jpg', '16905.jpg', '20767.jpg',
    '21593.jpg', '23303.jpg', '14642.jpg', '15992.jpg', '12727.jpg', '19064.jpg',
    '16415.jpg', '16944.jpg', '21761.jpg', '17631.jpg', '20110.jpg', '20038.jpg',
    '11909.jpg', '22641.jpg', '21024.jpg', '21372.jpg', '22164.jpg', '15419.jpg',
    '15023.jpg', '18613.jpg', '14328.jpg', '18002.jpg', '20761.jpg', '21235.jpg',
    '15774.jpg', '22316.jpg', '16099.jpg', '11788.jpg', '20923.jpg', '16461.jpg',
    '16705.jpg', '21014.jpg', '18355.jpg', '18379.jpg', '12032.jpg', '12453.jpg',
    '14990.jpg', '17465.jpg', '23313.jpg', '11876.jpg', '21720.jpg', '21664.jpg',
    '15294.jpg', '13209.jpg', '18757.jpg', '22814.jpg', '12466.jpg', '18964.jpg',
    '13665.jpg', '20620.jpg', '13580.jpg', '15033.jpg', '13577.jpg', '16996.jpg',
    '14686.jpg', '22575.jpg', '22879.jpg', '18249.jpg', '14720.jpg', '13164.jpg',
    '11761.jpg', '16419.jpg', '16627.jpg', '19566.jpg', '17788.jpg', '18122.jpg',
    '20811.jpg', '22249.jpg', '18963.jpg', '11904.jpg', '22836.jpg', '18897.jpg',
    '12544.jpg', '22053.jpg', '15647.jpg', '12905.jpg', '21487.jpg', '22443.jpg',
    '18266.jpg', '22986.jpg', '12479.jpg', '21803.jpg', '20919.jpg', '13151.jpg',
    '20507.jpg', '14765.jpg', '13031.jpg', '22781.jpg', '12678.jpg', '12653.jpg',
    '15654.jpg', '22923.jpg', '12269.jpg', '21125.jpg', '13847.jpg', '22100.jpg',
    '12968.jpg', '14279.jpg', '12529.jpg', '12444.jpg', '18929.jpg', '18199.jpg',
    '15513.jpg', '21587.jpg', '13505.jpg', '17906.jpg', '11772.jpg', '17923.jpg',
    '20617.jpg', '14917.jpg', '18762.jpg', '14637.jpg', '21736.jpg', '18605.jpg',
    '19307.jpg', '13729.jpg', '16281.jpg', '21827.jpg', '15291.jpg', '17609.jpg',
    '13850.jpg', '13330.jpg', '18313.jpg', '23259.jpg', '11953.jpg', '17303.jpg',
    '16166.jpg', '16191.jpg', '19826.jpg', '20932.jpg', '12709.jpg', '15490.jpg',
    '19890.jpg', '22797.jpg', '14510.jpg', '13687.jpg', '19236.jpg', '14984.jpg',
    '19815.jpg', '19049.jpg', '11987.jpg', '18373.jpg', '11859.jpg', '12298.jpg',
    '15416.jpg', '22531.jpg', '20896.jpg', '15823.jpg', '16676.jpg', '16425.jpg',
    '17792.jpg', '20540.jpg', '17814.jpg', '12427.jpg', '14276.jpg', '12900.jpg',
    '12134.jpg', '20301.jpg', '12632.jpg', '18760.jpg', '15369.jpg', '18579.jpg',
    '20529.jpg', '15107.jpg', '15325.jpg', '17737.jpg', '21375.jpg', '21620.jpg',
    '18805.jpg', '15915.jpg', '13602.jpg', '16477.jpg', '15535.jpg', '14225.jpg',
    '20151.jpg', '16199.jpg', '12620.jpg', '17972.jpg', '12639.jpg', '20215.jpg',
    '15280.jpg', '23382.jpg', '18334.jpg', '18705.jpg', '15525.jpg', '20347.jpg',
    '16210.jpg', '17248.jpg', '13339.jpg', '16866.jpg', '20147.jpg', '16307.jpg',
    '19042.jpg', '12209.jpg', '19746.jpg', '19564.jpg', '14241.jpg', '13855.jpg',
    '22522.jpg', '12169.jpg', '12015.jpg', '14447.jpg', '19942.jpg', '12016.jpg',
    '17434.jpg', '21028.jpg', '18763.jpg', '11869.jpg', '18918.jpg', '21311.jpg',
    '20251.jpg', '23118.jpg', '16504.jpg', '17043.jpg', '15832.jpg', '16907.jpg',
    '18024.jpg', '18346.jpg', '17217.jpg', '14228.jpg', '22540.jpg', '23401.jpg',
    '16457.jpg', '21722.jpg', '16100.jpg', '19468.jpg', '22985.jpg', '21708.jpg',
    '21502.jpg', '19158.jpg', '14489.jpg', '20565.jpg', '15600.jpg', '18615.jpg',
    '14752.jpg', '21510.jpg', '20608.jpg', '18036.jpg', '14146.jpg', '18871.jpg',
    '12749.jpg', '18023.jpg', '23335.jpg', '17714.jpg', '16614.jpg', '13475.jpg',
    '17977.jpg', '14840.jpg', '13020.jpg', '21712.jpg', '23012.jpg', '21450.jpg',
    '21569.jpg', '18795.jpg', '19983.jpg', '16182.jpg', '17641.jpg', '14651.jpg',
    '12987.jpg', '12755.jpg', '16456.jpg', '20494.jpg', '20920.jpg', '16931.jpg',
    '14753.jpg', '13102.jpg', '15315.jpg', '11748.jpg', '23351.jpg', '13515.jpg',
    '19986.jpg', '14838.jpg', '22010.jpg', '16845.jpg',
]

OUT = CFG["OUT"]; os.makedirs(OUT, exist_ok=True); T0 = time.time()
LOG_PATH = os.path.join(OUT, "run_log.txt")
def log(msg):
    line = f"[{(time.time() - T0) / 60:6.1f} min] {msg}"
    print(line, flush=True)
    with open(LOG_PATH, "a") as f: f.write(line + "\n")

SHARED = {"airplane": "plane", "plane": "plane", "ship": "ship", "boat": "ship", "storagetank": "storagetank", "storage-tank": "storagetank",
          "baseballfield": "baseball", "baseball-diamond": "baseball", "tenniscourt": "tennis", "tennis-court": "tennis",
          "basketballcourt": "basketball", "basketball-court": "basketball", "groundtrackfield": "groundtrack", "ground-track-field": "groundtrack",
          "harbor": "harbor", "bridge": "bridge", "vehicle": "vehicle", "large-vehicle": "vehicle", "small-vehicle": "vehicle"}
def attacker_count(model, kind, path):
    """Shared-class detections by a second detector on an image file (same JPEG as the primary measure)."""
    res = model.predict(source=path, save=False, conf=CFG["ATTACK_CONF"], verbose=False)
    n = 0
    for r in res:
        det = r.obb if kind == "obb" else r.boxes
        if det is None or len(det) == 0: continue
        for c in det.cls:
            if model.names.get(int(c), "") in SHARED: n += 1
    return n

def fidelity_extras(orig, prot):
    diff = np.any(orig != prot, axis=2); frac = float(diff.mean())
    if frac == 0.0: return 0.0, np.nan
    err = (orig.astype(np.float32) - prot.astype(np.float32)) ** 2
    mse = float(err[diff].mean())
    return frac, (10 * np.log10(255.0 ** 2 / mse) if mse > 0 else np.inf)

def _cpu_metrics(image, prot):
    mae, psnr_v, ssim_v = compute_utility_metrics(image, prot)
    frac, psnr_r = fidelity_extras(image, prot)
    return mae, psnr_v, ssim_v, frac, psnr_r

def _avail_bytes():
    try:
        import psutil; return psutil.virtual_memory().available
    except Exception:
        return 16e9
def _n_workers(pixels):
    """Metrics cost ~122 bytes/pixel per concurrent call; keep concurrent calls inside a RAM budget."""
    budget = min(8e9, 0.35 * _avail_bytes())
    return int(max(1, min(4, budget // (122 * pixels))))

def run_unified(tag, val_dir, sample, extract, build, residual_path_fn, attackers, budget_min, out_csv, max_mp=None, resume=True):
    rows, failures, excluded, done_names, done = [], [], [], set(), 0
    if resume and os.path.exists(out_csv):
        prev = pd.read_csv(out_csv)
        if len(prev):
            rows = prev.to_dict("records"); done_names = set(prev.image); done = int(prev.img_idx.max()) + 1
            log(f"[{tag}] RESUMING: {len(done_names)} images already scored")
    skipped_nodet = skipped_unreadable = 0; t_stage = time.time()
    tmp = os.path.join(OUT, f"_tmp_{tag}.jpg")
    for idx, fname in enumerate(sample):
        if fname in done_names: continue
        if (time.time() - t_stage) / 60 > budget_min:
            log(f"[{tag}] TIME BUDGET ({budget_min} min) reached after {len(done_names) + (done - len(done_names))} scored images; stopping stage"); break
        try:
            path = os.path.join(val_dir, fname)
            image = cv2.imread(path)
            if image is None: skipped_unreadable += 1; continue
            H, W = image.shape[:2]
            if max_mp and H * W > max_mp * 1e6:
                excluded.append({"image": fname, "height": H, "width": W}); log(f"[{tag}] {fname} skipped: {H}x{W} = {H * W / 1e6:.0f} MP > cap {max_mp} MP")
                del image; continue
            dets = extract(path)
            if not dets: skipped_nodet += 1; del image; continue
            n0 = len(dets)
            methods, rts, stats = build(image, dets, image.shape)
            with ThreadPoolExecutor(max_workers=_n_workers(H * W)) as ex:
                futs = {m: ex.submit(_cpu_metrics, image, p) for m, p in methods.items() if m != "original"}
                metrics = {m: f.result() for m, f in futs.items()}
            img_rows = []
            for mname, prot in methods.items():
                row = {"image": fname, "img_idx": done, "method": mname, "n_dets": n0}
                if mname == "original":
                    row.update(mae=0.0, psnr=np.nan, ssim=1.0); frac, psnr_r = 0.0, np.nan
                else:
                    mae, psnr_v, ssim_v, frac, psnr_r = metrics[mname]; row.update(mae=mae, psnr=psnr_v, ssim=ssim_v)
                cv2.imwrite(tmp, prot)                       # one JPEG per method, shared by the primary and attacker measures
                row["residual"] = residual_path_fn(tmp, n0)
                row["changed_frac"], row["psnr_changed_region"] = frac, psnr_r
                for aname, (model, kind) in attackers.items(): row[f"atk_{aname}"] = attacker_count(model, kind, tmp)
                row["runtime_ms"] = rts.get(mname, np.nan)
                if mname == "amapp_subregion_linear": row.update(stats)
                img_rows.append(row)
            rows.extend(img_rows); done += 1
            del methods, metrics, image; gc.collect()
        except Exception:
            failures.append({"image": fname, "trace": traceback.format_exc()}); log(f"[{tag}] image {fname} FAILED (see failures json); continuing")
            gc.collect()
            try:
                import torch; torch.cuda.empty_cache()
            except Exception: pass
        if done and done % 10 == 0:
            pd.DataFrame(rows).to_csv(out_csv, index=False)
            if done % 20 == 0:
                log(f"[{tag}] {done} scored ({idx + 1}/{len(sample)} visited)")
                try:
                    import torch; torch.cuda.empty_cache()
                except Exception: pass
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    with open(out_csv.replace(".csv", "_failures.json"), "w") as f: json.dump(failures, f, indent=1)
    with open(out_csv.replace(".csv", "_excluded_large.json"), "w") as f: json.dump(excluded, f, indent=1)
    log(f"[{tag}] DONE: scored {len(set(r['image'] for r in rows))}, no-detection {skipped_nodet}, unreadable {skipped_unreadable}, "
        f"over {max_mp} MP {len(excluded)}, failed {len(failures)} -> {out_csv}")
    return pd.DataFrame(rows)

def dior_extract(image_path):
    results = model_dior.predict(source=image_path, save=False, conf=CFG["CONF"], verbose=False)
    return [(DIOR_CLASSES[int(b.cls)], float(b.conf), b.xyxy[0].tolist()) for r in results if r.boxes is not None for b in r.boxes]

# ================================ DIOR adaptivity tests (cell F) ================================
import os, time, random, warnings
import numpy as np, pandas as pd, cv2
from skimage.metrics import peak_signal_noise_ratio as _psnr_fn

F_BUDGET_MIN = 30
UNIFORM_GRID = [("blur", 15), ("blur", 31), ("blur", 51), ("blur", 101),
                ("pix", 8), ("pix", 15), ("pix", 30), ("pix", 60)]
_TMPF = os.path.join(OUT, "_tmp_f.jpg")

def _apply_param(img, bbox, kind, p):             # in place; same maths as apply_operator
    x1, y1, x2, y2 = [int(v) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)
    if x2 <= x1 or y2 <= y1:
        return
    roi = img[y1:y2, x1:x2]
    if kind == "blur":
        k = p if p % 2 == 1 else p + 1
        img[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (k, k), 0)
    elif kind == "pix":
        small = cv2.resize(roi, (max(roi.shape[1] // p, 1), max(roi.shape[0] // p, 1)),
                           interpolation=cv2.INTER_LINEAR)
        img[y1:y2, x1:x2] = cv2.resize(small, (roi.shape[1], roi.shape[0]),
                                       interpolation=cv2.INTER_NEAREST)
    elif kind == "mask":
        img[y1:y2, x1:x2] = 0

def _redetect(img):
    cv2.imwrite(_TMPF, img)                        # same JPEG round trip as the primary metric
    res = model_dior.predict(source=_TMPF, save=False, conf=CFG["CONF"], verbose=False)
    return [(int(b.cls), b.xyxy[0].tolist()) for r in res if r.boxes is not None for b in r.boxes]

def _iou(a, b):
    ix1, iy1, ix2, iy2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def _survived(cid, bbox, new):
    return int(any(c == cid and _iou(bbox, nb) >= 0.5 for c, nb in new))

def _tier(s):
    return "high(>=0.7)" if s >= 0.7 else ("mid(0.4-0.6)" if s >= 0.4 else "low(<=0.3)")

def run_adaptivity(val_dir, out_prefix="ADAPT"):
    if globals().get("DIOR_SAMPLE"):                      # pinned list from cell G (same images as Tables 4-9)
        sample = [f for f in DIOR_SAMPLE if os.path.exists(os.path.join(val_dir, f))]
        log(f"[ADAPT] using pinned sample: {len(sample)} images")
    else:
        random.seed(CFG["SEED"])
        all_images = sorted(os.listdir(val_dir))
        sample = random.sample(all_images, min(CFG["SAMPLE"], len(all_images)))
        log("[ADAPT] WARNING: no pinned sample; this will not match Tables 4-9")
    img_rows, det_rows, done, t0 = [], [], 0, time.time()
    for idx, fname in enumerate(sample):
        if (time.time() - t0) / 60 > F_BUDGET_MIN:
            log(f"[ADAPT] time budget reached after {done} images"); break
        path = os.path.join(val_dir, fname)
        image = cv2.imread(path)
        if image is None:
            continue
        dets = dior_extract(path)
        if not dets:
            continue
        n0 = len(dets)
        amapp_img, scored = run_amapp_linear(image, dets, image.shape)     # identical to the main run
        # scored is sorted by MAPR, highest first: (cls, conf, bbox, mapr, vetoed)
        ops = [select_operator(s[3]) for s in scored]
        rr = random.Random(CFG["SEED"] * 1000 + done)
        shuf = ops[:]; rr.shuffle(shuf)
        rev = ops[::-1]
        protected = {"amapp": (amapp_img, ops)}
        for name, assigned in (("amapp_shuffled", shuf), ("amapp_reversed", rev)):
            p = image.copy()
            for s, op in zip(scored, assigned):
                p = apply_operator(p, s[2], op)
            protected[name] = (p, assigned)
        for kind, par in UNIFORM_GRID + [("mask", 0)]:
            p = image.copy()
            for s in scored:
                _apply_param(p, s[2], kind, par)
            protected["fixed_mask" if kind == "mask" else f"uniform_{kind}{par}"] = (p, None)
        comp = []
        for s in scored:
            comp.append((compute_exposure(s[2], image.shape),
                         compute_uniqueness(s[0], [(d[0], d[1], d[2]) for d in dets]),
                         compute_context(s[2], [(d[0], d[1], d[2]) for d in dets])))
        for mname, (prot, assigned) in protected.items():
            new = _redetect(prot)
            ps = float(_psnr_fn(image, prot, data_range=255)) if not np.array_equal(image, prot) else np.inf
            img_rows.append({"img_idx": done, "image": fname, "method": mname, "n_dets": n0,
                             "residual": len(new) / n0, "psnr": ps})
            for j, s in enumerate(scored):
                det_rows.append({"img_idx": done, "method": mname, "cls": s[0],
                                 "sens": SENSITIVITY_TABLE.get(s[0], 0.3), "conf": s[1], "mapr": s[3],
                                 "exposure": comp[j][0], "uniqueness": comp[j][1], "context": comp[j][2],
                                 "op_amapp": ops[j], "op_applied": (assigned[j] if assigned else mname),
                                 "survived": _survived(DIOR_CLASSES.index(s[0]), s[2], new)})
        done += 1
        if (idx + 1) % 25 == 0:
            log(f"[ADAPT] {idx + 1}/{len(sample)} visited, {done} scored")
    di, dd = pd.DataFrame(img_rows), pd.DataFrame(det_rows)
    di.to_csv(os.path.join(OUT, f"PERIMAGE_{out_prefix}.csv"), index=False)
    dd.to_csv(os.path.join(OUT, f"PERDET_{out_prefix}.csv"), index=False)
    log(f"[ADAPT] DONE: {done} images, {len(dd)} detection rows")
    return di, dd

# ----------------------------- analysis -----------------------------------
def _bsm(v, idx):
    v = np.asarray(v, float); v = np.where(np.isfinite(v), v, np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore"); return np.nanmean(v[idx], axis=1)

def _ci(a):
    lo, hi = np.nanpercentile(a, [2.5, 97.5]); return lo, hi

def analyze_adaptivity(di, dd, B=None):
    B = B or CFG["N_BOOT"]; rng = np.random.default_rng(CFG["SEED"]); L = []
    def out(s=""): print(s); L.append(s)
    n = di.img_idx.nunique(); idx = rng.integers(0, n, size=(B, n))
    Wr = di.pivot(index="img_idx", columns="method", values="residual")
    Wp = di.pivot(index="img_idx", columns="method", values="psnr")
    out("=" * 100); out(f"TEST 1 - uniform-strength sweep vs AMAPP (n={n} images, paired 95% bootstrap CIs)"); out("=" * 100)
    rr = {m: 1 - _bsm(Wr[m], idx) for m in Wr.columns}; ps = {m: _bsm(Wp[m], idx) for m in Wp.columns}
    rows = []
    for m in Wr.columns:
        fin = Wp[m].replace([np.inf, -np.inf], np.nan)
        rows.append({"method": m, "RR_%": 100 * (1 - Wr[m].mean()), "RR_lo": 100 * _ci(rr[m])[0], "RR_hi": 100 * _ci(rr[m])[1],
                     "PSNR": fin.mean(), "PSNR_lo": _ci(ps[m])[0], "PSNR_hi": _ci(ps[m])[1],
                     "dRR_vs_amapp": 100 * (rr[m] - rr["amapp"]).mean(), "dPSNR_vs_amapp": np.nanmean(ps[m] - ps["amapp"])})
    T1 = pd.DataFrame(rows).sort_values("PSNR", ascending=False); out(T1.round(2).to_string(index=False))
    a = T1[T1.method == "amapp"].iloc[0]
    near = T1[(T1.method.str.startswith("uniform") | (T1.method == "fixed_mask")) & ((T1.PSNR - a.PSNR).abs() <= 2.0)]
    out(f"\nuniform configs within +-2 dB of AMAPP's PSNR ({a.PSNR:.2f}): " +
        (", ".join(f"{r.method} RR {r['RR_%']:.1f}% / {r.PSNR:.2f} dB" for _, r in near.iterrows()) or "none"))
    out(f"AMAPP: RR {a['RR_%']:.1f}% at {a.PSNR:.2f} dB")
    T1.to_csv(os.path.join(OUT, "ADAPT_test1_frontier.csv"), index=False)

    # reproduction check against the main run (Table 4): AMAPP whole-object and fixed masking on the same 286 images
    amr = T1[T1.method == "amapp"].iloc[0]; fmr = T1[T1.method == "fixed_mask"].iloc[0]
    ok = n == 286 and abs(amr['RR_%'] - 75.52) < 0.1 and abs(amr.PSNR - 28.93) < 0.02 and abs(fmr['RR_%'] - 93.36) < 0.1
    out(f"REPRODUCTION CHECK vs main run (expect n=286, AMAPP 75.5% / 28.93 dB, fixed mask 93.4%): measured n={n}, AMAPP {amr['RR_%']:.1f}% / {amr.PSNR:.2f} dB, "
        f"fixed mask {fmr['RR_%']:.1f}% -> {'OK' if ok else 'MISMATCH - do not use these numbers'}")
    out("\n" + "=" * 100); out("TEST 2 - protection rate by sensitivity tier (per detection: 1 - still re-found at IoU>=0.5)"); out("=" * 100)
    dd = dd.copy(); dd["tier"] = dd.sens.map(_tier)
    tiers = ["high(>=0.7)", "mid(0.4-0.6)", "low(<=0.3)"]
    cnt = dd[dd.method == "amapp"].groupby("tier").size()
    out("detections per tier: " + ", ".join(f"{t} {int(cnt.get(t, 0))}" for t in tiers))
    methods = ["amapp", "amapp_shuffled", "amapp_reversed", "uniform_blur31", "uniform_pix15", "uniform_blur101", "uniform_pix60", "fixed_mask"]
    rows = []
    for m in methods:
        sub = dd[dd.method == m]; rec = {"method": m}; bs = {}
        for t in tiers:
            s = sub[sub.tier == t]
            tot = s.groupby("img_idx").size().reindex(range(n), fill_value=0).to_numpy(float)
            sv = s.groupby("img_idx").survived.sum().reindex(range(n), fill_value=0).to_numpy(float)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore"); b = 1 - sv[idx].sum(1) / tot[idx].sum(1)
            bs[t] = b; lo, hi = _ci(b)
            rec[t] = f"{100 * (1 - sv.sum() / max(tot.sum(), 1)):.1f}% [{100 * lo:.0f},{100 * hi:.0f}]"
        sel = bs[tiers[0]] - bs[tiers[2]]; lo, hi = _ci(sel)
        rec["high_minus_low_pts"] = f"{100 * np.nanmean(sel):+.1f} [{100 * lo:+.0f},{100 * hi:+.0f}]"
        rows.append(rec)
    T2 = pd.DataFrame(rows); out(T2.to_string(index=False)); T2.to_csv(os.path.join(OUT, "ADAPT_test2_tiers.csv"), index=False)
    out("(high_minus_low > 0 means the high-sensitivity objects are protected more than the low-sensitivity ones)")

    out("\n" + "=" * 100); out("TEST 3 - operator mix, exposure and components (AMAPP, per detection)"); out("=" * 100)
    A = dd[dd.method == "amapp"]
    out("operator counts: " + ", ".join(f"{k} {v} ({100 * v / len(A):.1f}%)" for k, v in A.op_amapp.value_counts().items()))
    mix = pd.crosstab(A.tier, A.op_amapp, normalize="index").mul(100).round(1).reindex(tiers); out("operator mix by tier (% of detections):"); out(mix.to_string())
    e = A.exposure
    out(f"exposure: mean {e.mean():.3f}, std {e.std():.3f}, min {e.min():.3f}, p05 {e.quantile(.05):.3f}, median {e.median():.3f}, share >= 0.999: {100 * (e >= 0.999).mean():.1f}%")
    out(f"MAPR: min {A.mapr.min():.2f}, p25 {A.mapr.quantile(.25):.2f}, median {A.mapr.median():.2f}, p75 {A.mapr.quantile(.75):.2f}, max {A.mapr.max():.2f}")
    out("component means by tier:"); out(A.groupby("tier")[["sens", "conf", "exposure", "uniqueness", "context", "mapr"]].mean().round(3).reindex(tiers).to_string())
    byc = A.groupby("cls").agg(n=("mapr", "size"), sens=("sens", "mean"), conf=("conf", "mean"), mapr=("mapr", "mean")).round(2).sort_values("n", ascending=False)
    ops = pd.crosstab(A.cls, A.op_amapp, normalize="index").mul(100).round(0)
    out("by class (n, sensitivity, mean confidence, mean MAPR, operator mix %):"); out(byc.join(ops).to_string())
    with open(os.path.join(OUT, "RESULTS_SUMMARY_ADAPT.txt"), "w") as f:
        f.write("\n".join(L))
    return T1, T2


# ================================ DOTA ================================
def _box_op_inplace(img, bbox, operator):          # in-place twin of apply_operator (same maths, no full-image copy)
    if operator in ("preserve", "none"): return img
    x1, y1, x2, y2 = [int(v) for v in bbox]; x1, y1 = max(0, x1), max(0, y1); x2, y2 = min(img.shape[1], x2), min(img.shape[0], y2)
    if x2 <= x1 or y2 <= y1: return img
    roi = img[y1:y2, x1:x2]
    if operator == "blur": img[y1:y2, x1:x2] = cv2.GaussianBlur(roi, (31, 31), 0)
    elif operator == "pixelate":
        small = cv2.resize(roi, (max(roi.shape[1] // 15, 1), max(roi.shape[0] // 15, 1)), interpolation=cv2.INTER_LINEAR)
        img[y1:y2, x1:x2] = cv2.resize(small, (roi.shape[1], roi.shape[0]), interpolation=cv2.INTER_NEAREST)
    elif operator == "mask": img[y1:y2, x1:x2] = 0
    else: raise ValueError(f"Unknown operator: {operator}")
    return img

def _polygon_op_inplace(img, corners, operator):   # same pixels as apply_operator_to_mask(polygon_to_mask(...)), ROI-local and in place
    if operator in ("preserve", "none"): return img
    H, W = img.shape[:2]; pts = np.array(corners, dtype=np.int32).reshape(-1, 2)
    x1 = max(int(pts[:, 0].min()), 0); y1 = max(int(pts[:, 1].min()), 0)
    x2 = min(int(pts[:, 0].max()) + 1, W); y2 = min(int(pts[:, 1].max()) + 1, H)
    if x2 <= x1 or y2 <= y1: return img
    pm = np.zeros((y2 - y1, x2 - x1), dtype=np.uint8); cv2.fillPoly(pm, [pts - np.array([x1, y1], dtype=np.int32)], 1); pm = pm.astype(bool)
    ys, xs = np.where(pm)
    if len(ys) == 0: return img
    ty1, ty2, tx1, tx2 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    roi = img[y1 + ty1:y1 + ty2, x1 + tx1:x1 + tx2]; roi_mask = pm[ty1:ty2, tx1:tx2]
    if operator == "blur": t = cv2.GaussianBlur(roi, (31, 31), 0)
    elif operator == "pixelate":
        small = cv2.resize(roi, (max(roi.shape[1] // 15, 1), max(roi.shape[0] // 15, 1)), interpolation=cv2.INTER_LINEAR)
        t = cv2.resize(small, (roi.shape[1], roi.shape[0]), interpolation=cv2.INTER_NEAREST)
    elif operator == "mask": t = np.zeros_like(roi)
    else: raise ValueError(f"Unknown operator: {operator}")
    roi[roi_mask] = t[roi_mask]
    return img

def apply_operator_obb(image, corners, operator):  # modifies `image` in place; every caller passes its own private copy
    if OBB_MASK_MODE == "aabb": return _box_op_inplace(image, obb_corners_to_aabb(corners), operator)
    return _polygon_op_inplace(image, corners, operator)

def dota_extract(image_path):
    results = model_dota.predict(source=image_path, save=False, conf=CFG["CONF"], verbose=False); dets = []
    for r in results:
        obb = r.obb
        if obb is None or len(obb) == 0: continue
        for i in range(len(obb)):
            corners = obb.xyxyxyxy[i].tolist()
            dets.append((DOTA_CLASSES[int(obb.cls[i])], float(obb.conf[i]), obb_corners_to_aabb(corners), corners))
    return dets

def dota_build(image, dets, shape):
    m = {"original": image.copy(), "global_blur": baseline_global_blur(image.copy()),
         "fixed_mask": baseline_fixed_mask_obb(image, dets), "single_attribute": baseline_single_attribute_obb(image, dets)}
    rts = {}
    t = time.perf_counter(); m["amapp_whole_linear"], _ = run_amapp_linear_obb(image, dets, shape); rts["amapp_whole_linear"] = (time.perf_counter() - t) * 1000
    t = time.perf_counter(); m["amapp_subregion_linear"], plog = run_amapp_subregion_linear_obb(image, dets, shape, SAM_MODEL); rts["amapp_subregion_linear"] = (time.perf_counter() - t) * 1000
    elig = [e for e in plog if e["class"] in SUBREGION_CLASSES]
    return m, rts, {"n_eligible": len(elig), "n_subregion": sum(e["path"] == "subregion" for e in elig),
                    "n_fallback": sum(e["path"] == "whole_object_fallback_no_masks" for e in elig)}

def residual_obb_path(path, n0):                   # same measure as compute_residual_detectability_obb, on an already-written JPEG
    res = model_dota.predict(source=path, save=False, conf=0.4, verbose=False)
    return sum(len(r.obb) for r in res if r.obb is not None) / n0 if n0 > 0 else 0.0

# ================================ analysis helpers (from the main run) ================================
B = CFG["N_BOOT"]
rng = np.random.default_rng(CFG["SEED"])
S = []                                   # summary lines -> printed and saved
def out(line=""):
    print(line); S.append(line)

def bsmean(v, idx):
    """Bootstrap means of v over resample index matrix idx; non-finite values are ignored."""
    v = np.asarray(v, float)
    v = np.where(np.isfinite(v), v, np.nan)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return np.nanmean(v[idx], axis=1)

def ci(a):
    lo, hi = np.nanpercentile(a, [2.5, 97.5])
    return lo, hi

def pu_from(rr, psnr):
    return 0.5 * rr + 0.5 * np.minimum(psnr / PSNR_NORMALIZATION_CAP, 1.0)

REF = {  # numbers already in your results doc: (risk reduction %, PSNR)
    "DIOR": {"global_blur": (75.1, 21.94), "fixed_mask": (93.4, 18.46),
             "single_attribute": (43.5, 22.99), "amapp_subregion_linear": (67.9, 30.31),
             "amapp_whole_linear": (74.8, 29.82)},
    "DOTA": {"global_blur": (39.2, 22.91), "fixed_mask": (80.8, 22.28),
             "single_attribute": (12.7, 33.32), "amapp_subregion_linear": (47.2, 33.08)},
}
PAIRS = [("amapp_subregion_linear", "fixed_mask"), ("amapp_subregion_linear", "global_blur"),
         ("amapp_subregion_linear", "single_attribute"), ("amapp_subregion_linear", "amapp_whole_linear")]

def analyze_main(df, label):
    out("=" * 100); out(f"{label}: main results with 95% bootstrap CIs  (B={B})"); out("=" * 100)
    W = {c: df.pivot(index="img_idx", columns="method", values=c)
         for c in ["residual", "psnr", "ssim", "mae", "changed_frac", "psnr_changed_region"]}
    n = len(W["residual"])
    idx = rng.integers(0, n, size=(B, n))       # ONE resample matrix -> everything is paired
    methods = list(W["residual"].columns)
    rr_bs, ps_bs, rr_pt, ps_pt, rows = {}, {}, {}, {}, []
    for m in methods:
        rr_bs[m] = 1 - bsmean(W["residual"][m], idx)
        rr_pt[m] = 1 - W["residual"][m].mean()
        fin = W["psnr"][m].replace([np.inf, -np.inf], np.nan)
        ps_bs[m] = bsmean(W["psnr"][m], idx)
        ps_pt[m] = fin.mean()
        n_inf = int(np.isinf(W["psnr"][m]).sum())
        rlo, rhi = ci(rr_bs[m]); plo, phi = ci(ps_bs[m]) if m != "original" else (np.nan, np.nan)
        pu_pt = pu_from(rr_pt[m], ps_pt[m]) if m != "original" else np.nan
        rows.append({"method": m, "N": n,
                     "RR_%": 100 * rr_pt[m], "RR_lo": 100 * rlo, "RR_hi": 100 * rhi,
                     "PSNR": ps_pt[m], "PSNR_lo": plo, "PSNR_hi": phi, "n_inf_psnr": n_inf,
                     "SSIM": W["ssim"][m].mean(), "MAE": W["mae"][m].mean(),
                     "changed_px_%": 100 * W["changed_frac"][m].mean(),
                     "PSNR_changed_region": W["psnr_changed_region"][m].replace([np.inf, -np.inf], np.nan).mean(),
                     "PU": pu_pt})
    T = pd.DataFrame(rows)
    out(T.round(3).to_string(index=False))
    T.to_csv(os.path.join(OUT, f"CI_main_{label}.csv"), index=False)

    out(f"\n-- paired differences (A minus B), same {n} images in every method --")
    for a, b in PAIRS:
        if a in methods and b in methods:
            d_rr = 100 * (rr_bs[a] - rr_bs[b]); d_ps = ps_bs[a] - ps_bs[b]
            l1, h1 = ci(d_rr); l2, h2 = ci(d_ps)
            out(f"{a} - {b}:  dRR = {100 * (rr_pt[a] - rr_pt[b]):+.1f} pts [{l1:+.1f}, {h1:+.1f}]"
                f"   dPSNR = {ps_pt[a] - ps_pt[b]:+.2f} dB [{l2:+.2f}, {h2:+.2f}]")

    out("\n-- self-check vs numbers already in your results doc (RR %, PSNR) --")
    for m, (rr_ref, ps_ref) in REF[label].items():
        if m in methods:
            flag = "" if (abs(100 * rr_pt[m] - rr_ref) <= 3 and abs(ps_pt[m] - ps_ref) <= 1.0) else "   <-- MISMATCH"
            out(f"{m:26s} measured {100 * rr_pt[m]:5.1f}% / {ps_pt[m]:5.2f} dB   reference {rr_ref:5.1f}% / {ps_ref:5.2f} dB{flag}")

    if "n_eligible" in df.columns:
        s = df[df.method == "amapp_subregion_linear"]
        te, ts, tf = s.n_eligible.sum(), s.n_subregion.sum(), s.n_fallback.sum()
        out(f"\nsub-region eligible detections: {int(te)} total, {int(ts)} via sub-region path, {int(tf)} fell back to whole-object")
    rt = df.groupby("method").runtime_ms.mean().dropna()
    out("mean runtime per image (ms): " + ", ".join(f"{k}={v:.0f}" for k, v in rt.items()))

    for ac in [c for c in df.columns if c.startswith("atk_")]:
        Wa = df.pivot(index="img_idx", columns="method", values=ac)
        valid = Wa["original"] > 0
        nv = int(valid.sum())
        out(f"\n-- independent attacker '{ac[4:]}': images where it finds >=1 shared-class object on the unprotected image: {nv}/{n} --")
        if nv < 20:
            out("   too few valid images for a meaningful estimate; skipped"); continue
        Wv = Wa[valid]
        idx2 = rng.integers(0, nv, size=(B, nv))
        orig_cnt = Wv["original"].to_numpy(float)
        arows = []
        for m in Wv.columns:
            if m == "original":
                continue
            ratio = Wv[m].to_numpy(float) / orig_cnt
            bs = 1 - bsmean(ratio, idx2); lo, hi = ci(bs)
            arows.append({"method": m, "attacker_RR_%": 100 * (1 - ratio.mean()), "lo": 100 * lo, "hi": 100 * hi,
                          "primary_RR_%_same_images": 100 * (1 - W["residual"].loc[valid, m].mean())})
        out(pd.DataFrame(arows).round(1).to_string(index=False))
    out("")
    return T


# ================================ RUN ================================
def _run_all():
    global SAM_MODEL
    try:
        import ultralytics, torch
        log(f"ENV: ultralytics {ultralytics.__version__}, torch {torch.__version__}, cuda available: {torch.cuda.is_available()}")
    except Exception as e: log(f"could not read environment versions: {e}")
    log(f"settings: OBB_MASK_MODE={OBB_MASK_MODE}, DOTA_MAX_MP={DOTA_MAX_MP}, DOTA_BUDGET_MIN={DOTA_BUDGET_MIN}, storage-tank sub-region={INCLUDE_STORAGE_TANK_SUBREGION}")
    f_summary = os.path.join(OUT, "RESULTS_SUMMARY_ADAPT.txt")
    if RUN_F and not (SKIP_F_IF_DONE and os.path.exists(f_summary)):
        try:
            di, dd = run_adaptivity(CFG["DIOR_VAL_DIR"]); analyze_adaptivity(di, dd)
        except Exception: log("ADAPT stage crashed:\n" + traceback.format_exc())
    else: log("[ADAPT] skipped (disabled, or already done in this session)")
    if RUN_DOTA:
        try:
            SAM_MODEL = get_sam_model(); log("SAM loaded OK")
            if INCLUDE_STORAGE_TANK_SUBREGION:
                SUBREGION_CLASSES.add("storage-tank"); SENSITIVITY_TABLE_PARTS["storage-tank"] = SENSITIVITY_TABLE_PARTS["storagetank"]
                _orig = globals().get("_orig_label_parts") or label_parts
                globals()["_orig_label_parts"] = _orig
                globals()["label_parts"] = lambda masks, bbox, cn: _orig(masks, bbox, "storagetank" if cn == "storage-tank" else cn)
            random.seed(CFG["SEED"]); allimg = sorted(os.listdir(CFG["DOTA_VAL_DIR"]))     # sorted -> same sample in every session
            sample = random.sample(allimg, min(CFG["SAMPLE"], len(allimg)))
            df = run_unified("DOTA", CFG["DOTA_VAL_DIR"], sample, dota_extract, dota_build, residual_obb_path,
                             {"dior_xfer": (model_dior, "box")}, DOTA_BUDGET_MIN, os.path.join(OUT, "PERIMAGE_dota_v3.csv"), max_mp=DOTA_MAX_MP)
            S.clear()
            ex_path = os.path.join(OUT, "PERIMAGE_dota_v3_excluded_large.json"); n_ex = len(json.load(open(ex_path))) if os.path.exists(ex_path) else 0
            out(f"DOTA sample: {len(sample)} drawn (seed {CFG['SEED']}, sorted listing); scored {df.image.nunique() if len(df) else 0}; skipped for size (> {DOTA_MAX_MP} MP): {n_ex}; oriented-mask mode: {OBB_MASK_MODE}")
            out("(the DOTA reference numbers in the self-check come from an earlier run on a different image sample; they are shown for orientation only)")
            if len(df): analyze_main(df, "DOTA")
            with open(os.path.join(OUT, "RESULTS_SUMMARY_DOTA_v3.txt"), "w") as f: f.write("\n".join(S))
        except Exception: log("DOTA stage crashed:\n" + traceback.format_exc())
    keep = ["RESULTS_SUMMARY_ADAPT.txt", "RESULTS_SUMMARY_DOTA_v3.txt", "run_log.txt", "PERIMAGE_ADAPT.csv", "PERDET_ADAPT.csv", "PERIMAGE_dota_v3.csv",
            "PERIMAGE_dota_v3_failures.json", "PERIMAGE_dota_v3_excluded_large.json", "ADAPT_test1_frontier.csv", "ADAPT_test2_tiers.csv", "CI_main_DOTA.csv"]
    with zipfile.ZipFile(os.path.join(OUT, "AMAPP_final_outputs.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        for k in keep:
            if os.path.exists(os.path.join(OUT, k)): z.write(os.path.join(OUT, k), k)
    log("ALL DONE. Download AMAPP_final_outputs.zip, and paste RESULTS_SUMMARY_ADAPT.txt + RESULTS_SUMMARY_DOTA_v3.txt back to Claude.")

if globals().get("FINAL_AUTORUN", True):
    _run_all()

In [ ]:
"""
EXPORT CELL for amapp-updated.ipynb  -- paste as a NEW LAST CELL (does not change any existing code).

BEFORE RUNNING
  1. In the last existing cell set  FINAL_AUTORUN = False  (otherwise it re-runs the 80-minute DOTA stage).
  2. Run all cells above so the functions exist (dior_extract, dior_build, dota_extract, dota_build,
     get_sam_model, DIOR_SAMPLE, CFG). DOTA_VAL_DIR must exist in this session.

WHAT IT DOES (one pass, resumable)
  For the same images as the paper (DIOR: pinned DIOR_SAMPLE; DOTA: seed-42 sample, 40 MP cap, images with
  at least one detection) it saves, per image:
     __ref__/            unmodified original file (copy)            -> D_ref for LAE-DINO
     original/           JPEG round trip of the original            -> noise floor
     global_blur/ fixed_mask/ single_attribute/ amapp_whole_linear/ amapp_subregion_linear/
  Protected files are written with cv2.imwrite exactly as run_unified does, so they are the very JPEGs used for
  the primary measure. Output goes to /tmp (about 5-8 GB for DOTA), not /kaggle/working.

SUCCESS CHECK (printed at the end): DIOR 286 images and DOTA 259 images exported, 6 files per image.
"""
import os, gc, random, shutil, traceback
import cv2

EXPORT_ROOT = "/tmp/protected-images"
EXPECTED = {"DIOR": 286, "DOTA": 259}
METHODS_SAVED = ["original", "global_blur", "fixed_mask", "single_attribute", "amapp_whole_linear", "amapp_subregion_linear"]

SAM_MODEL = get_sam_model()          # the build functions read this global


def _done(ds, stem):
    return all(os.path.exists(os.path.join(EXPORT_ROOT, ds, m, stem + ".jpg")) for m in METHODS_SAVED) and \
           any(f.startswith(stem + ".") for f in os.listdir(os.path.join(EXPORT_ROOT, ds, "__ref__")))


def _export(ds, val_dir, sample, extract, build, max_mp=None):
    for m in METHODS_SAVED + ["__ref__"]:
        os.makedirs(os.path.join(EXPORT_ROOT, ds, m), exist_ok=True)
    n_ok = n_nodet = n_big = n_fail = 0
    for i, fname in enumerate(sample):
        stem = os.path.splitext(fname)[0]
        try:
            if _done(ds, stem): n_ok += 1; continue
            path = os.path.join(val_dir, fname)
            image = cv2.imread(path)
            if image is None: continue
            H, W = image.shape[:2]
            if max_mp and H * W > max_mp * 1e6: n_big += 1; continue          # same cap as run_unified
            dets = extract(path)
            if not dets: n_nodet += 1; continue                                # same rule as run_unified
            methods, _, _ = build(image, dets, image.shape)
            for mname, prot in methods.items():
                cv2.imwrite(os.path.join(EXPORT_ROOT, ds, mname, stem + ".jpg"), prot)   # same default JPEG quality
            shutil.copy(path, os.path.join(EXPORT_ROOT, ds, "__ref__", fname))
            n_ok += 1
            del methods, image; gc.collect()
        except Exception:
            n_fail += 1; print(ds, fname, "FAILED\n", traceback.format_exc())
            gc.collect()
        if (i + 1) % 25 == 0: print(f"[{ds}] {i + 1}/{len(sample)} visited, {n_ok} exported", flush=True)
    print(f"[{ds}] exported {n_ok}; no detection {n_nodet}; over size cap {n_big}; failed {n_fail}")
    return n_ok


# ---- DIOR: the pinned sample used for Tables 4-9 ----
dior_sample = [f for f in DIOR_SAMPLE if os.path.exists(os.path.join(CFG["DIOR_VAL_DIR"], f))]
n_dior = _export("DIOR", CFG["DIOR_VAL_DIR"], dior_sample, dior_extract, dior_build)

# ---- DOTA: same sampling code as _run_all ----
random.seed(CFG["SEED"])
allimg = sorted(os.listdir(CFG["DOTA_VAL_DIR"]))
dota_sample = random.sample(allimg, min(CFG["SAMPLE"], len(allimg)))
n_dota = _export("DOTA", CFG["DOTA_VAL_DIR"], dota_sample, dota_extract, dota_build, max_mp=DOTA_MAX_MP)

# ---- success check ----
for ds, n in (("DIOR", n_dior), ("DOTA", n_dota)):
    print(f"CHECK {ds}: {n} exported, expected {EXPECTED[ds]} ->", "OK" if n == EXPECTED[ds] else "MISMATCH (stop and look)")

In [ ]:
import os, re, sys, shutil, subprocess, urllib.request

TMP = "/tmp"
REPO = f"{TMP}/LAE-DINO"
REPO_DIR = f"{REPO}/mmdetection_lae"
W_DIR = f"{TMP}/laedino"
ENVDIR = f"{TMP}/lae_env_recent"
PY = f"{ENVDIR}/bin/python"
WORKER_PATH = f"{TMP}/laedino_worker.py"
WEIGHTS = {
    "laedino_pretrain.pth": "1EiR8KtNRYIeOfvtIe9C82cQk_uOMIQ8U",
    "laedino_ft_dior.pth": "1Q35PUdzUHLIM22ozm_ccgJzXXtfMXSOm",
    "laedino_ft_dotav2.pth": "1rxDO2QGNWw8WiDKRTqqVrK-JE4rE3VVT",
}
CFG0 = "lae_dino_swin-t_pretrain_LAE-1M.py"
CONFIGS = [CFG0, "lae_dino_swin-t_finetune_DIOR.py", "lae_dino_swin-t_finetune_DOTA.py"]


def step(msg):
    print(f"\n=== {msg} ===", flush=True)


def run(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)
    r = subprocess.run(cmd, shell=isinstance(cmd, str), **kw)
    if r.returncode != 0:
        raise SystemExit(f"STOPPED: command failed with code {r.returncode}: {cmd}")


def find_root():
    if os.path.isdir("/tmp/protected-images/DIOR"):          # same session as the export
        return "/tmp/protected-images"
    for base in ("/kaggle/input", "/kaggle/input/datasets"):  # or the added dataset
        for dp, dn, _ in os.walk(base):
            if "DIOR" in dn and "DOTA" in dn:
                return dp
            if dp.count("/") > 6:
                dn[:] = []
    raise SystemExit("STOPPED: images not found in /tmp/protected-images or /kaggle/input.")


ROOT = find_root()

# Environment for the isolated Python:
#  MPLBACKEND=Agg      -> Kaggle's notebook backend crashes matplotlib inside the venv (your last error)
#  PYTHONPATH          -> repo used directly (no pip -e), also drops Kaggle's own PYTHONPATH
#  PYTHONNOUSERSITE    -> notebook's user-site packages cannot leak into the venv
#  CUDA_VISIBLE_DEVICES=1 -> if the session has 2 GPUs, use the 2nd (the 1st may still hold the export's memory)
ENV = dict(os.environ, PYTHONPATH=REPO_DIR, PROTECTED_ROOT=ROOT, MPLBACKEND="Agg", PYTHONNOUSERSITE="1")
try:
    n_gpu = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
except Exception:
    n_gpu = 0
if n_gpu >= 2:
    ENV["CUDA_VISIBLE_DEVICES"] = "1"

SMOKE = ("import torch, mmcv, mmcv.ops, mmdet; from mmengine.config import Config; from mmdet.utils import register_all_modules; "
         "register_all_modules(); Config.fromfile(r'{cfg}'); "
         "print('SMOKE OK torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'mmcv', mmcv.__version__, 'mmdet', mmdet.__version__); "
         "assert torch.cuda.is_available(), 'no GPU: switch the Kaggle accelerator on'")


def build_env():
    smoke = [PY, "-c", SMOKE.format(cfg=f"{REPO_DIR}/configs/lae_dino/{CFG0}")]
    if os.path.exists(PY) and subprocess.run(smoke, cwd=REPO_DIR, env=ENV).returncode == 0:
        return                                                   # already built and working
    if not os.path.exists(PY):
        run(["uv", "venv", "--seed", "--python", "3.10", ENVDIR])
    uv = ["uv", "pip", "install", "--python", PY]
    run(uv + ["torch==2.1.0", "torchvision==0.16.0", "--index-url", "https://download.pytorch.org/whl/cu118"])
    run(uv + ["numpy<2", "setuptools<70", "opencv-python-headless", "mmengine", "pandas", "scipy", "matplotlib",
              "pycocotools", "terminaltables", "shapely", "six", "pillow", "nltk"])   # setuptools<70: mmengine needs pkg_resources
    run(uv + ["mmcv==2.1.0", "-f", "https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html"])
    req = f"{REPO_DIR}/requirements/multimodal.txt"
    if os.path.exists(req):
        run(uv + ["-r", req])
    run(uv + ["emoji", "ddd-dataset", "transformers==4.38.2", "numpy<2", "setuptools<70"])   # re-pin: nothing may upgrade these
    # NO editable install: the repo is used through PYTHONPATH instead
    # the repo imports packages that are not in its requirements (clip, open_clip, ...). Install them up front.
    run(uv + ["wheel", "setuptools<70", "ftfy", "regex", "tqdm", "einops", "safetensors",
              "huggingface_hub<1.0", "timm==0.9.16", "open_clip_torch==2.24.0"])
    run(uv + ["--no-build-isolation", "git+https://github.com/openai/CLIP.git"])
    run(uv + ["transformers==4.38.2", "numpy<2", "setuptools<70", "huggingface_hub<1.0"])   # re-pin again after the new packages
    # anything else still missing: install it and retry
    pip_name = {"clip": "git+https://github.com/openai/CLIP.git", "open_clip": "open_clip_torch==2.24.0",
                "cv2": "opencv-python-headless", "PIL": "pillow", "sklearn": "scikit-learn", "yaml": "pyyaml"}
    for _ in range(8):
        r = subprocess.run(smoke, cwd=REPO_DIR, env=ENV, capture_output=True, text=True)
        print(r.stdout, r.stderr[-1500:], flush=True)
        if r.returncode == 0:
            return
        m = re.search(r"No module named '([\w\.]+)'", r.stderr)
        if not m:
            break                                                # a different kind of error: stop and show it
        mod = m.group(1).split(".")[0]
        print(f"--- missing module '{mod}', installing ---", flush=True)
        extra = ["--no-build-isolation"] if mod == "clip" else []
        run(uv + extra + [pip_name.get(mod, mod)])
    raise SystemExit("STOPPED: smoke test still failing. Paste the lines printed above.")


def setup():
    step("0. checks")
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=10)
    except Exception as e:
        raise SystemExit(f"STOPPED: no Internet ({e}). Settings -> Internet -> On.")
    for ds, n in (("DIOR", 286), ("DOTA", 259)):
        d = f"{ROOT}/{ds}/__ref__"
        have = len(os.listdir(d)) if os.path.isdir(d) else 0
        if have != n:
            raise SystemExit(f"STOPPED: {d} has {have} files, expected {n}.")
    free = shutil.disk_usage(TMP).free / 1e9
    if free < 10:
        raise SystemExit(f"STOPPED: only {free:.1f} GB free in /tmp, need about 10.")
    print(f"images OK at {ROOT}; GPUs seen: {n_gpu}; free /tmp: {free:.0f} GB")

    step("1. repo, weights, BERT")
    run([sys.executable, "-m", "pip", "install", "-q", "gdown", "uv"])
    if not os.path.isdir(REPO_DIR):
        run(["git", "clone", "--depth", "1", "https://github.com/jaychempan/LAE-DINO", REPO])
    missing = [c for c in CONFIGS if not os.path.exists(f"{REPO_DIR}/configs/lae_dino/{c}")]
    if missing:
        raise SystemExit(f"STOPPED: config files not found: {missing}")
    mim = f"{REPO_DIR}/mmdet/.mim"                               # lets 'mmdet::' config bases resolve without pip -e
    os.makedirs(mim, exist_ok=True)
    if not os.path.exists(f"{mim}/configs"):
        os.symlink(f"{REPO_DIR}/configs", f"{mim}/configs")
    os.makedirs(W_DIR, exist_ok=True)
    for fn, gid in WEIGHTS.items():
        p = f"{W_DIR}/{fn}"
        if not os.path.exists(p) or os.path.getsize(p) < 10_000_000:
            run([sys.executable, "-m", "gdown", gid, "-O", p])
        if os.path.getsize(p) < 10_000_000:
            raise SystemExit(f"STOPPED: {fn} did not download (Google Drive quota?). Wait an hour and rerun this cell.")
    bert = f"{REPO}/weights/bert-base-uncased"
    if not os.path.exists(f"{bert}/config.json"):
        from huggingface_hub import snapshot_download
        snapshot_download("google-bert/bert-base-uncased", local_dir=bert)
    alt = f"{REPO_DIR}/weights"
    os.makedirs(alt, exist_ok=True)
    if not os.path.exists(f"{alt}/bert-base-uncased"):
        os.symlink(bert, f"{alt}/bert-base-uncased")

    step("2. environment (slow the first time, then skipped)")
    build_env()


def main():
    setup()
    step("3. re-detection (resumable)")
    open(WORKER_PATH, "w").write(WORKER)
    p = subprocess.Popen([PY, "-u", WORKER_PATH], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ENV)
    for line in p.stdout:
        if "Inference" in line and "\u2501" in line:          # drop the per-image progress bars
            continue
        print(line, end="", flush=True)
    rc = p.wait()
    print(f"\nworker exit code {rc} (0 = finished and noise-floor check passed)")
    print("Outputs: /kaggle/working/laedino_redetect/summary.csv and table_laedino.md")


WORKER = r'''
import os, sys, json, csv, glob, time
import numpy as np
import cv2

ROOT = os.environ["PROTECTED_ROOT"]
OUT_DIR = "/kaggle/working/laedino_redetect"
ORIGINAL = "original"
METHODS = ["original", "global_blur", "fixed_mask", "single_attribute", "amapp_whole_linear", "amapp_subregion_linear"]
REF = "__ref__"
EXPECTED_N = {"DIOR": 286, "DOTA": 259}
THRESHOLDS = [0.25, 0.40]
SEED, N_BOOT = 42, 2000
N_DRY = 8

REPO_DIR = "/tmp/LAE-DINO/mmdetection_lae"
W = "/tmp/laedino"
CFG_DIR = REPO_DIR + "/configs/lae_dino"
MODELS = {
    "laedino_pretrained": {"config": CFG_DIR + "/lae_dino_swin-t_pretrain_LAE-1M.py", "weights": W + "/laedino_pretrain.pth",
                           "datasets": ["DIOR", "DOTA"]},
    "laedino_ft_dior": {"config": CFG_DIR + "/lae_dino_swin-t_finetune_DIOR.py", "weights": W + "/laedino_ft_dior.pth",
                        "datasets": ["DIOR"]},
    "laedino_ft_dotav2": {"config": CFG_DIR + "/lae_dino_swin-t_finetune_DOTA.py", "weights": W + "/laedino_ft_dotav2.pth",
                          "datasets": ["DOTA"]},
}
PROMPTS = {
    "DIOR": {"airplane": "plane", "ship": "ship", "storage tank": "storage tank", "baseball field": "baseball field",
             "tennis court": "tennis court", "basketball court": "basketball court",
             "ground track field": "ground track field", "harbor": "harbour", "bridge": "bridge", "vehicle": "vehicle"},
    "DOTA": {"plane": "plane", "ship": "ship", "storage tank": "storage tank", "baseball diamond": "baseball field",
             "tennis court": "tennis court", "basketball court": "basketball court",
             "ground track field": "ground track field", "harbor": "harbour", "bridge": "bridge",
             "large vehicle": "vehicle", "small vehicle": "vehicle"},
}
EXTS = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")


def list_images(dataset, method):
    d = os.path.join(ROOT, dataset, method)
    return {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(os.path.join(d, "*")) if p.lower().endswith(EXTS)}


def read(path):
    img = cv2.imread(path)
    assert img is not None, f"cannot read image: {path}"
    return img


_inferencers = {}
_patched = False


def patch_inferencer():
    """The repo's DetInferencer builds the TRAINING-set annotations (../data/LAE-FOD/...json) just to read class names.
    We do not need them (the prompt carries the class names), so skip that step."""
    global _patched
    if _patched:
        return
    from mmdet.apis import DetInferencer
    from mmdet.evaluation import get_classes

    def _load_weights_to_model(self, model, checkpoint, cfg):
        if checkpoint is not None:
            try:
                from mmengine.runner.checkpoint import _load_checkpoint_to_model
                _load_checkpoint_to_model(model, checkpoint)
            except ImportError:
                model.load_state_dict(checkpoint.get("state_dict", checkpoint), strict=False)
        model.dataset_meta = {"classes": get_classes("coco")}   # placeholder, unused with text prompts
        model.cfg = cfg

    DetInferencer._load_weights_to_model = _load_weights_to_model

    # the test dataset in these configs is a wrapper (ConcatDataset): the pipeline sits one level deeper
    orig_init_pipeline = DetInferencer._init_pipeline

    def _init_pipeline(self, cfg):
        ds = cfg.test_dataloader.dataset
        if "pipeline" not in ds:
            d = ds
            while "pipeline" not in d:
                if "datasets" in d:
                    d = d["datasets"][0]
                elif "dataset" in d:
                    d = d["dataset"]
                else:
                    raise RuntimeError("no pipeline found in test_dataloader.dataset")
            ds["pipeline"] = d["pipeline"]
        return orig_init_pipeline(self, cfg)

    DetInferencer._init_pipeline = _init_pipeline
    _patched = True


def detect(model_name, dataset, img_bgr):
    from mmdet.apis import DetInferencer
    patch_inferencer()
    if model_name not in _inferencers:
        os.chdir(REPO_DIR)
        m = MODELS[model_name]
        _inferencers[model_name] = DetInferencer(model=m["config"], weights=m["weights"], device="cuda", show_progress=False)
    names = list(PROMPTS[dataset].keys())
    prompt = ". ".join(names) + "."
    out = _inferencers[model_name](img_bgr, texts=prompt, custom_entities=True, no_save_pred=True,
                                   return_vis=False, pred_score_thr=min(THRESHOLDS), print_result=False)
    pred = out["predictions"][0]
    return [(PROMPTS[dataset][names[int(l)]], float(s)) for l, s in zip(pred["labels"], pred["scores"])
            if 0 <= int(l) < len(names)]


def counts(dets):
    return {str(t): int(sum(1 for _, s in dets if s >= t)) for t in THRESHOLDS}


def bootstrap_rr(ratios, rng):
    r = np.asarray(ratios, float)
    if len(r) == 0:
        return float("nan"), float("nan"), float("nan")
    rr = 100 * (1 - r.mean())
    bs = [100 * (1 - r[rng.integers(0, len(r), len(r))].mean()) for _ in range(N_BOOT)]
    return rr, np.percentile(bs, 2.5), np.percentile(bs, 97.5)


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    cache_path = os.path.join(OUT_DIR, "per_image.jsonl")

    images = {}
    for ds in EXPECTED_N:
        per_method = {m: list_images(ds, m) for m in METHODS + [REF]}
        stems = set.intersection(*[set(v) for v in per_method.values()])
        assert len(stems) == EXPECTED_N[ds], f"{ds}: found {len(stems)} common images, expected {EXPECTED_N[ds]}"
        images[ds] = (sorted(stems), per_method)
    for name, m in MODELS.items():
        assert os.path.exists(m["config"]) and os.path.exists(m["weights"]), f"missing files for {name}"

    # dry run: must find objects, otherwise the whole run would silently produce zeros
    ds0, name0 = list(EXPECTED_N)[0], list(MODELS)[0]
    t0, total = time.time(), 0
    for i, stem in enumerate(images[ds0][0][:N_DRY]):
        dets = detect(name0, ds0, read(images[ds0][1][REF][stem]))
        total += sum(1 for _, s in dets if s >= min(THRESHOLDS))
        if i < 3:
            print("dry run", ds0, stem, sorted(dets, key=lambda x: -x[1])[:5], flush=True)
    print(f"dry run: {total} detections (score >= {min(THRESHOLDS)}) in {N_DRY} images, "
          f"{(time.time() - t0) / N_DRY:.1f} s/image incl. model load", flush=True)
    assert total > 0, "dry run found NO objects: the prompt/label mapping or the model call is wrong. Paste the lines above."

    done = {}
    if os.path.exists(cache_path):
        for line in open(cache_path):
            r = json.loads(line); done[(r["model"], r["dataset"], r["method"], r["image"])] = r
    t_start = time.time()
    with open(cache_path, "a") as f:
        for model_name, m in MODELS.items():
            for ds in m["datasets"]:
                stems, per_method = images[ds]
                for k, stem in enumerate(stems):
                    for method in [REF] + METHODS:
                        key = (model_name, ds, method, stem)
                        if key in done:
                            continue
                        rec = {"model": model_name, "dataset": ds, "method": method, "image": stem,
                               "counts": counts(detect(model_name, ds, read(per_method[method][stem])))}
                        f.write(json.dumps(rec) + "\n"); done[key] = rec
                    if k % 25 == 0:
                        f.flush(); print(model_name, ds, f"{k + 1}/{len(stems)}", f"{(time.time() - t_start) / 60:.0f} min elapsed", flush=True)

    rng = np.random.default_rng(SEED)
    rows = []
    for model_name, m in MODELS.items():
        for ds in m["datasets"]:
            stems = images[ds][0]
            for t in THRESHOLDS:
                ref = {s: done[(model_name, ds, REF, s)]["counts"][str(t)] for s in stems}
                use = [s for s in stems if ref[s] >= 1]
                for method in METHODS:
                    ratios = [done[(model_name, ds, method, s)]["counts"][str(t)] / ref[s] for s in use]
                    rr, lo, hi = bootstrap_rr(ratios, rng)
                    rows.append({"model": model_name, "dataset": ds, "threshold": t, "method": method,
                                 "n_images_used": len(use), "RR": round(rr, 1), "CI_low": round(lo, 1), "CI_high": round(hi, 1)})
    with open(os.path.join(OUT_DIR, "summary.csv"), "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

    cols = [(mn, ds) for mn, m in MODELS.items() for ds in m["datasets"]]
    lines = ["| Method | " + " | ".join(f"{ds}, {mn}" for mn, ds in cols) + " |", "|---|" + "---|" * len(cols)]
    for method in METHODS[1:]:
        cells = []
        for mn, ds in cols:
            r = next(x for x in rows if x["model"] == mn and x["dataset"] == ds and x["threshold"] == 0.25 and x["method"] == method)
            cells.append(f"{r['RR']:.1f} ({r['CI_low']:.1f} to {r['CI_high']:.1f})")
        lines.append(f"| {method} | " + " | ".join(cells) + " |")
    open(os.path.join(OUT_DIR, "table_laedino.md"), "w").write("\n".join(lines))
    print("\n".join(lines))

    bad = [r for r in rows if r["method"] == ORIGINAL and not abs(r["RR"]) <= 3]   # NaN counts as bad
    print("CHECK original noise floor within +/-3 points:", "OK" if not bad else f"FAILED {bad}")
    sys.exit(0 if not bad else 1)


main()
'''

main()